In [ ]:
from ollama import chat, Client

"""
Configure register selection, LLM parameters, API access, and automatic recovery settings for specification processing.
"""

### CHANGE HERE ###
LIST_REGISTER_WANTED = [
    'hidelegh',
    'hpmcounter20',
]
IS_USING_EVERYTHING = False # Set this to True if you want to use all possible registers
### CHANGE HERE ###

MODEL = 'qwen3.8:27b'
IS_LOCAL = False
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = ''
CLIENT = Client(host='https://ollama.com', timeout=1800, headers={'Authorization': API_KEY})
WIDTH = 120

RESET_PROGRESS = False

# Automatic LLM recovery.
AUTO_RETRY = True
CHECKPOINT_EVERY = 10
MAX_RETRIES = 10
RETRY_WAIT_SECONDS = 30
RUNNER_STOP_TIMEOUT_SECONDS = 60

print('====\nDONE\n====')

In [1]:
# 1 - CREATE - 1_REGISTER_ALL.txt - START

"""
Extract and normalize RISC-V CSR metadata, including register names, addresses, privilege modes, widths, and bit fields, using riscv-unified-db and riscv-opcodes.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '2_INPUT/riscv-unified-db/spec/std/isa/csr'
FOLDER_INPUT_OTHER = '2_INPUT/riscv-opcodes'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_BLOCK_SORT_KEY(STRING_BLOCK: str):
    MATCH = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    STRING_NAME = MATCH.group(1).strip() if MATCH else STRING_BLOCK
    return FUNCTION_NATURAL_SORT_KEY(STRING_NAME)

def FUNCTION_GET_PRIV_AND_WIDTH(STRING_REGISTER: str):
    if STRING_REGISTER.startswith(('seed', 'ssp')):
        return 'U-mode', 'XLEN'
    STRING_FIRST_CHAR = STRING_REGISTER[0].lower()
    if STRING_FIRST_CHAR == 'h':
        return 'HS-mode', 'SXLEN'
    if STRING_FIRST_CHAR == 'd':
        return 'D-mode', 'MXLEN'
    if STRING_FIRST_CHAR == 'm':
        return 'M-mode', 'MXLEN'
    if STRING_FIRST_CHAR == 's':
        return 'S-mode', 'SXLEN'
    return 'U-mode', 'XLEN'

def FUNCTION_OVERRIDE_SPECIAL_FIELDS(STRING_BLOCK: str) -> str:
    MATCH_REGISTER = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    if not MATCH_REGISTER:
        return STRING_BLOCK

    STRING_REGISTER = MATCH_REGISTER.group(1).strip().lower()

    if STRING_REGISTER == 'mseccfg':
        STRING_FIELDS = STRING_FIELDS_MSECCFG
    elif STRING_REGISTER == 'senvcfg':
        STRING_FIELDS = STRING_FIELDS_SENVCFG
    elif STRING_REGISTER == 'hstatus':
        STRING_FIELDS = STRING_FIELDS_HSTATUS
    else:
        return STRING_BLOCK

    if re.search(r'^FIELDS:\s*.*$', STRING_BLOCK, flags=re.M):
        STRING_BLOCK = re.sub(
            r'^FIELDS:\s*.*$',
            f'FIELDS: {STRING_FIELDS}',
            STRING_BLOCK,
            flags=re.M
        )
    else:
        STRING_BLOCK += f'\nFIELDS: {STRING_FIELDS}'

    return STRING_BLOCK

def FUNCTION_IS_ODD_PMPCFG(STRING_REGISTER: str):
    MATCH_VALUE = re.fullmatch(r'pmpcfg(\d+)', STRING_REGISTER.lower())
    return bool(MATCH_VALUE) and int(MATCH_VALUE.group(1)) % 2 == 1

def FUNCTION_OVERRIDE_MODE(STRING_BLOCK: str) -> str:
    MATCH_REGISTER = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    if not MATCH_REGISTER:
        return STRING_BLOCK

    STRING_REGISTER = MATCH_REGISTER.group(1).strip().lower()

    if re.search(r'^MODE:\s*S-mode\s*$', STRING_BLOCK, flags=re.M):
        STRING_BLOCK = re.sub(
            r'^MODE:\s*S-mode\s*$',
            'MODE: HS-mode',
            STRING_BLOCK,
            flags=re.M
        )

    if STRING_REGISTER in SET_REGISTER_HS_MODE:
        STRING_BLOCK = re.sub(
            r'^MODE:\s*.*$',
            'MODE: HS-mode',
            STRING_BLOCK,
            flags=re.M
        )

    return STRING_BLOCK
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_OUTPUT = []
SET_REGISTER_YAML = set()
MAP_ENTRY_MISSING = {}
STRING_FIELDS_MSECCFG = '33..32=PMM, 10..10=MLPE, 9..9=SSEED, 8..8=USEED, 2..2=RLB, 1..1=MMWP, 0..0=MML'
STRING_FIELDS_SENVCFG = '63..34=WPRI, 33..32=PMM, 31..8=WPRI, 7..7=CBZE, 6..6=CBCFE, 5..4=CBIE, 3..3=SSE, 2..2=LPE, 1..1=WPRI, 0..0=FIOM'
STRING_FIELDS_HSTATUS = '63..50=WPRI, 49..48=HUPMM, 47..34=WPRI, 33..32=VSXL, 31..23=WPRI, 22..22=VTSR, 21..21=VTW, 20..20=VTVM, 19..18=WPRI, 17..12=VGEIN, 11..10=WPRI, 9..9=HU, 8..8=SPVP, 7..7=SPV, 6..6=GVA, 5..5=VSBE, 4..0=WPRI'
SET_REGISTER_HS_MODE = {
    'vsatp',
    'vscause',
    'vsctrctl',
    'vsepc',
    'vsie',
    'vsieh',
    'vsip',
    'vsiph',
    'vsireg',
    'vsireg2',
    'vsireg3',
    'vsireg4',
    'vsireg5',
    'vsireg6',
    'vsiselect',
    'vsscratch',
    'vsstatus',
    'vstimecmp',
    'vstimecmph',
    'vstopei',
    'vstopi',
    'vstval',
    'vstvec',
}
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_PATH, _, EACH_LIST_FILE in os.walk(FOLDER_INPUT):
    for EACH_FILE_NAME in EACH_LIST_FILE:
        if not EACH_FILE_NAME.endswith('.yaml'):
            continue
        FILE_PATH = os.path.join(EACH_PATH, EACH_FILE_NAME)
        with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            LIST_LINE = FILE_HANDLE.readlines()
        STRING_CURRENT = ''
        IS_FIELD = False
        LIST_EXTENSION = []
        LIST_BIT = []
        LIST_LOCATION = []
        INDENT_FIELDS = None
        IS_DEFINED_BY = False
        IS_ALL_OF = False
        HAD_FIELDS = False
        for EACH_LINE in LIST_LINE:
            STRING_LINE_STRIPPED = EACH_LINE.strip()
            if IS_FIELD:
                INDENT_LINE = len(EACH_LINE) - len(EACH_LINE.lstrip(' '))
                if INDENT_FIELDS is not None and INDENT_LINE <= INDENT_FIELDS and STRING_LINE_STRIPPED:
                    IS_FIELD = False
                if IS_FIELD:
                    if (
                        STRING_LINE_STRIPPED.endswith(':')
                        and not STRING_LINE_STRIPPED.startswith('-')
                        and ':' in STRING_LINE_STRIPPED
                        and INDENT_FIELDS is not None
                        and INDENT_LINE == INDENT_FIELDS + 2
                    ):
                        STRING_FIELD_NAME = STRING_LINE_STRIPPED[:-1].strip()
                        if STRING_FIELD_NAME not in ('fields', 'definedBy', 'allOf'):
                            LIST_BIT.append(STRING_FIELD_NAME)
                        continue
                    if STRING_LINE_STRIPPED.startswith(('location:', 'location_rv64:')):
                        LIST_LOCATION.append(EACH_LINE.split(':', 1)[1].strip())
                        continue
                    if STRING_LINE_STRIPPED.startswith(('affectedBy:', 'definedBy:')):
                        LIST_EXTENSION.append(EACH_LINE.split(':', 1)[1].strip())
                        continue
                    continue
            if (
                STRING_LINE_STRIPPED.startswith('name:')
                and not IS_DEFINED_BY
                and 'REGISTER:' not in STRING_CURRENT
            ):
                STRING_REGISTER = EACH_LINE.split(':', 1)[1].strip()
                STRING_CURRENT += f'REGISTER: {STRING_REGISTER}\n'
            elif STRING_LINE_STRIPPED.startswith('address:'):
                STRING_CURRENT += f"NUMBER: {EACH_LINE.split(':', 1)[1].strip().lower()}\n"
            elif STRING_LINE_STRIPPED.startswith('priv_mode:'):
                STRING_PRIV = EACH_LINE.split(':', 1)[1].strip()
                if STRING_CURRENT.startswith('REGISTER: h') and STRING_PRIV == 'S':
                    STRING_PRIV = 'HS'
                STRING_CURRENT += f'MODE: {STRING_PRIV}-mode\n'
            elif STRING_LINE_STRIPPED.startswith('length:'):
                STRING_CURRENT += f"WIDTH: {EACH_LINE.split(':', 1)[1].strip()}\n"
            elif STRING_LINE_STRIPPED.startswith('definedBy:'):
                LIST_EXTENSION.append(EACH_LINE.split(':', 1)[1].strip())
                IS_DEFINED_BY = True
            elif IS_DEFINED_BY and not STRING_LINE_STRIPPED.startswith('fields:'):
                if STRING_LINE_STRIPPED.startswith('allOf:'):
                    IS_ALL_OF = True
                elif IS_ALL_OF and STRING_LINE_STRIPPED.startswith('-'):
                    LIST_EXTENSION.append(EACH_LINE.split('-', 1)[1].strip())
                elif STRING_LINE_STRIPPED.startswith('name:'):
                    LIST_EXTENSION.append(EACH_LINE.split('name:', 1)[1].strip())
            elif STRING_LINE_STRIPPED.startswith('fields:'):
                IS_ALL_OF = False
                IS_DEFINED_BY = False
                if STRING_LINE_STRIPPED == 'fields: {}':
                    LIST_EXTENSION = [EACH_EXTENSION for EACH_EXTENSION in LIST_EXTENSION if EACH_EXTENSION]
                    if not LIST_EXTENSION:
                        LIST_EXTENSION.append('-')
                    STRING_CURRENT += 'FIELDS: -\nEXTENSIONS: -'
                    break
                IS_FIELD = True
                HAD_FIELDS = True
                INDENT_FIELDS = len(EACH_LINE) - len(EACH_LINE.lstrip(' '))
        if HAD_FIELDS:
            LIST_BIT_FINAL = []
            if LIST_LOCATION and LIST_LOCATION[0].startswith('0'):
                ITER_PAIR = zip(reversed(LIST_LOCATION), reversed(LIST_BIT))
                for EACH_LOCATION, EACH_NAME in ITER_PAIR:
                    if '-' in EACH_LOCATION:
                        STRING_A, STRING_B = EACH_LOCATION.split('-')
                        LIST_BIT_FINAL.append(f'{STRING_B}..{STRING_A}={EACH_NAME}')
                    else:
                        LIST_BIT_FINAL.append(f'{EACH_LOCATION}..{EACH_LOCATION}={EACH_NAME}')
            else:
                ITER_PAIR = zip(LIST_LOCATION, LIST_BIT)
                for EACH_LOCATION, EACH_NAME in ITER_PAIR:
                    if '-' in EACH_LOCATION:
                        STRING_A, STRING_B = EACH_LOCATION.split('-')
                        LIST_BIT_FINAL.append(f'{STRING_A}..{STRING_B}={EACH_NAME}')
                    else:
                        LIST_BIT_FINAL.append(f'{EACH_LOCATION}..{EACH_LOCATION}={EACH_NAME}')
            if LIST_BIT_FINAL:
                STRING_CURRENT += f"FIELDS: {', '.join(LIST_BIT_FINAL)}\n"
            else:
                STRING_CURRENT += 'FIELDS: -\n'
            if LIST_EXTENSION:
                LIST_EXTENSION = [EACH_EXTENSION for EACH_EXTENSION in LIST_EXTENSION if EACH_EXTENSION]
                LIST_EXTENSION_FLAT = []
                for EACH_EXTENSION in LIST_EXTENSION:
                    if EACH_EXTENSION.startswith('[') and EACH_EXTENSION.endswith(']'):
                        LIST_EXTENSION_FLAT.extend(EACH_PART.strip() for EACH_PART in EACH_EXTENSION[1:-1].split(','))
                    else:
                        LIST_EXTENSION_FLAT.append(EACH_EXTENSION)
                LIST_EXTENSION = sorted(set(EACH_X.split('#')[0].strip() for EACH_X in LIST_EXTENSION_FLAT))
                STRING_CURRENT += 'EXTENSIONS: -'
            else:
                STRING_CURRENT += 'EXTENSIONS: -'
        if STRING_CURRENT:
            if re.search(r'(?m)^\s*MODE:\s*D-mode\s*$', STRING_CURRENT):
                continue
            STRING_CURRENT = FUNCTION_OVERRIDE_SPECIAL_FIELDS(STRING_CURRENT)
            MATCH_REGISTER = re.search(r'^REGISTER:\s*([^\n]+)', STRING_CURRENT, flags=re.M)
            STRING_REGISTER = MATCH_REGISTER.group(1).strip() if MATCH_REGISTER else ''
            if FUNCTION_IS_ODD_PMPCFG(STRING_REGISTER):
                continue
            LIST_OUTPUT.append(STRING_CURRENT)
            if STRING_REGISTER:
                SET_REGISTER_YAML.add(STRING_REGISTER)

LIST_FILE_CSV = [EACH_FILE_NAME for EACH_FILE_NAME in os.listdir(FOLDER_INPUT_OTHER) if EACH_FILE_NAME.startswith('csrs') and EACH_FILE_NAME.endswith('.csv')]

for EACH_FILE_NAME in sorted(LIST_FILE_CSV, key=FUNCTION_NATURAL_SORT_KEY):
    FILE_PATH = os.path.join(FOLDER_INPUT_OTHER, EACH_FILE_NAME)
    with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        for EACH_LINE in FILE_HANDLE:
            STRING_LINE = EACH_LINE.strip()
            if not STRING_LINE or STRING_LINE.startswith('#'):
                continue
            LIST_PART = STRING_LINE.split(',')
            if len(LIST_PART) < 2:
                continue
            STRING_HEX = LIST_PART[0].strip().lower()
            STRING_REGISTER = LIST_PART[1].strip().strip('"')
            if FUNCTION_IS_ODD_PMPCFG(STRING_REGISTER):
                continue
            if STRING_REGISTER not in SET_REGISTER_YAML:
                STRING_PRIV, STRING_WIDTH = FUNCTION_GET_PRIV_AND_WIDTH(STRING_REGISTER)
                if STRING_PRIV == 'D-mode':
                    continue
                MAP_ENTRY_MISSING[STRING_REGISTER] = {'hex': STRING_HEX, 'priv': STRING_PRIV, 'width': STRING_WIDTH}

for EACH_REGISTER in list(MAP_ENTRY_MISSING):
    if EACH_REGISTER.endswith('h'):
        STRING_BASE = EACH_REGISTER[:-1]
        if STRING_BASE in MAP_ENTRY_MISSING:
            MAP_ENTRY_MISSING[STRING_BASE]['width'] = '64'
            MAP_ENTRY_MISSING[EACH_REGISTER]['width'] = '32'

for EACH_REGISTER, EACH_INFO in sorted(MAP_ENTRY_MISSING.items()):
    LIST_OUTPUT.append(f'REGISTER: {EACH_REGISTER}\n' f"NUMBER: {EACH_INFO['hex']}\n" f"MODE: {EACH_INFO['priv']}\n" f"WIDTH: {EACH_INFO['width']}\n" f'FIELDS: -\n' f'EXTENSIONS: -')

LIST_OUTPUT = [
    FUNCTION_OVERRIDE_MODE(EACH_BLOCK)
    for EACH_BLOCK in LIST_OUTPUT
]

LIST_OUTPUT.sort(key=FUNCTION_BLOCK_SORT_KEY)

LIST_OUTPUT = [
    EACH_BLOCK
    for EACH_BLOCK in LIST_OUTPUT
    if not FUNCTION_IS_ODD_PMPCFG(re.search(r'^REGISTER:\s*([^\n]+)', EACH_BLOCK, flags=re.M).group(1).strip())
]

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_REGISTER_ALL.txt - END

0.05 second(s)

====
DONE
====


In [2]:
# 2 - CREATE - 2_REGISTER_SOME.txt - START

"""
Select RISC-V CSRs and their metadata based on user configuration, exclude unsupported registers, and include corresponding base registers for selected virtual register aliases.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_FIND_V_ALIAS(TEXT_VALUE: str):
    LIST_NAME = [EACH_MATCH.group(1).strip() for EACH_MATCH in re.finditer(r'^REGISTER:\s*([^\n]+)', TEXT_VALUE, flags=re.M)]
    SET_NAME = set(LIST_NAME)
    SET_SEEN = set()
    LIST_OUTPUT = []
    for EACH_NAME in LIST_NAME:
        if (EACH_NAME.startswith('v') and len(EACH_NAME) > 1 and EACH_NAME[1:] in SET_NAME and EACH_NAME not in SET_SEEN):
            LIST_OUTPUT.append(EACH_NAME)
            SET_SEEN.add(EACH_NAME)
    return LIST_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_NOT_IMPLEMENTED = ['sireg', 'sireg2', 'sireg3', 'sireg4', 'sireg5', 'sireg6', 'vsireg', 'vsireg2', 'vsireg3', 'vsireg4', 'vsireg5', 'vsireg6', 'jvt', 'mnstatus', 'mncause', 'tcontrol', 'mireg', 'mireg2', 'mireg3', 'mireg4', 'mireg5', 'mireg6', 'mtopei', *[f'hpmcounter{i}' for i in range(19, 32)], *[f'hpmcounter{i}h' for i in range(3, 32)], *[f'mhpmcounter{i}' for i in range(19, 32)], *[f'mhpmcounter{i}h' for i in range(3, 32)], 'pmpcfg1', 'pmpcfg3', 'pmpcfg5', 'pmpcfg7', 'pmpcfg9', 'pmpcfg11', 'pmpcfg13', 'pmpcfg15', 'mctrctl', 'mcyclecfg', 'mcyclecfgh', 'mcycleh', 'medelegh', 'menvcfgh', *[f'mhpmevent{i}h' for i in range(3, 32)], 'midelegh', 'mieh', 'minstretcfg', 'minstretcfgh', 'minstreth', 'mintstatus', 'miph', 'mnepc', 'mnscratch', 'mnxti', 'mscratchcsw', 'mscratchcswl', 'mseccfgh', 'mstatush', 'mtvt', 'mvienh', 'mviph', 'hcontext', 'hedelegh', 'henvcfgh', 'hidelegh', 'htimedeltah', 'hvienh', 'hviph', 'hviprio1h', 'hviprio2h', 'mscontext', 'scontext', 'sctrctl', 'sctrdepth', 'sctrstatus', 'sieh', 'sintstatus', 'siph', 'snxti', 'srmcfg', 'sscratchcsw', 'sscratchcswl', 'stimecmph', 'stopei', 'stvt', 'cycleh', 'instreth', 'timeh', 'uintstatus', 'unxti', 'uscratchcsw', 'uscratchcswl', 'utvt', 'vsieh', 'vsiph', 'vstimecmph', 'vstopei', 'scountovf', 'stimecmp', 'vstimecmp', 'hstateen0h', 'hstateen1h', 'hstateen2h', 'hstateen3h', 'mstateen0h', 'mstateen1h', 'mstateen2h', 'mstateen3h', 'aes32dsi', 'aes32dsmi', 'aes32esi', 'aes32esmi']
# VARIABLE - FINISH

# CODE - BEGIN
LIST_REGISTER_WANTED = [
    EACH_REGISTER
    for EACH_REGISTER in LIST_REGISTER_WANTED
    if EACH_REGISTER not in LIST_NOT_IMPLEMENTED
]

if not LIST_REGISTER_WANTED:
    LIST_REGISTER_WANTED.append('tdata1')

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_ALL = FILE_HANDLE.read()

SET_V_ALIAS = set(FUNCTION_FIND_V_ALIAS(TEXT_ALL))

for EACH_V_ALIAS in (SET_V_ALIAS & set(LIST_REGISTER_WANTED)):
    LIST_REGISTER_WANTED.append(EACH_V_ALIAS[1:])

LIST_REGISTER_WANTED.sort(key=FUNCTION_NATURAL_SORT_KEY)

LIST_LINE = TEXT_ALL.splitlines(True)
SET_NOT_IMPLEMENTED = {EACH.lower() for EACH in LIST_NOT_IMPLEMENTED}

if IS_USING_EVERYTHING:
    LIST_OUT = []
    EACH_INDEX_LINE = 0
    while EACH_INDEX_LINE < len(LIST_LINE):
        EACH_LINE = LIST_LINE[EACH_INDEX_LINE]
        STRIPPED = EACH_LINE.strip()
        if STRIPPED.startswith('REGISTER:'):
            NAME = STRIPPED.split(':', 1)[1].strip().lower()
            if NAME in SET_NOT_IMPLEMENTED:
                EACH_INDEX_LINE += 1
                while (
                    EACH_INDEX_LINE < len(LIST_LINE)
                    and not LIST_LINE[EACH_INDEX_LINE].strip().startswith('REGISTER:')
                ):
                    EACH_INDEX_LINE += 1
                continue
        LIST_OUT.append(EACH_LINE)
        EACH_INDEX_LINE += 1
    TEXT_FINAL = ''.join(LIST_OUT)
    TEXT_FINAL = re.sub(r'\n{3,}', '\n\n', TEXT_FINAL).strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_FINAL)
else:
    LIST_WANTED = sorted({EACH_WANTED.lower() for EACH_WANTED in LIST_REGISTER_WANTED})
    LIST_SELECTED = []
    for EACH_WANTED in LIST_WANTED:
        STRING_TARGET = f'REGISTER: {EACH_WANTED}'
        for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
            if EACH_LINE.strip() == STRING_TARGET:
                LIST_SELECTED.append(''.join(EACH_L.strip() + '\n' for EACH_L in LIST_LINE[EACH_INDEX_LINE:EACH_INDEX_LINE + 6]).rstrip('\n'))
                break
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write('\n\n'.join(LIST_SELECTED))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_REGISTER_SOME.txt - END

0.00 second(s)

====
DONE
====


In [ ]:
# 3 - CREATE - 3_REGISTER_BOOLEAN.txt - START

"""
Identify relevant specification sections for each selected RISC-V CSR using keyword matching and LLM-based relevance classification, including related register families and aliases.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client, ResponseError
import hashlib, json, os, re, subprocess
from pathlib import Path
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '3_SPECIFICATION/2_SPLIT'
FILE_INPUT = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_PROMPT = '5_REGISTER/2_HISTORY/1_HISTORY_REGISTER_BOOLEAN.txt'
FILE_ALL = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
FILE_CHECKPOINT = '5_REGISTER/3_WHICH_LOOP/WHICH_LOOP_BOOLEAN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'"))

def FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    STRING_ERROR = str(ERROR).lower()
    LIST_RETRY_KEYWORD = ['cuda', 'gpu', 'launch timed out', 'ggml_backend_cuda', 'cudastreamsynchronize', 'cannot access memory', 'runner']
    if STATUS_CODE is not None and STATUS_CODE >= 500:
        return True
    return any(EACH_KEYWORD in STRING_ERROR for EACH_KEYWORD in LIST_RETRY_KEYWORD)

def FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    return STATUS_CODE in (408, 429) or (STATUS_CODE is not None and STATUS_CODE >= 500)

def FUNCTION_STOP_LOCAL_RUNNER(MODEL):
    print(f'[RECOVERY] Stopping Ollama runner for {MODEL!r}...', flush=True)
    try:
        COMPLETED_PROCESS = subprocess.run(['ollama', 'stop', MODEL], check=False, capture_output=True, text=True, timeout=RUNNER_STOP_TIMEOUT_SECONDS)
        if COMPLETED_PROCESS.returncode != 0:
            STRING_ERROR = COMPLETED_PROCESS.stderr.strip()
            if STRING_ERROR:
                print(f'[RECOVERY] ollama stop returned {COMPLETED_PROCESS.returncode}: {STRING_ERROR}', flush=True)
    except Exception as ERROR:
        print(f'[RECOVERY] Could not stop runner cleanly: {ERROR}', flush=True)

def FUNCTION_LLM_CALL(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT):
    NUMBER_ATTEMPT = 0
    while True:
        NUMBER_ATTEMPT += 1
        try:
            if IS_LOCAL:
                return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
            return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
        except (ResponseError, ConnectionError) as ERROR:
            IS_RETRYABLE = FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR) if IS_LOCAL else FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR)
            if not AUTO_RETRY or not IS_RETRYABLE or NUMBER_ATTEMPT >= MAX_RETRIES:
                print(f'[RECOVERY] Giving up after attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}.', flush=True)
                raise
            STRING_MODE = 'LOCAL' if IS_LOCAL else 'ONLINE'
            print(f'[RECOVERY] {STRING_MODE} request failed on attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}: {ERROR}', flush=True)
            if IS_LOCAL:
                FUNCTION_STOP_LOCAL_RUNNER(MODEL)
            print(f'[RECOVERY] Waiting {RETRY_WAIT_SECONDS} seconds, then retrying the SAME request automatically...', flush=True)
            time.sleep(RETRY_WAIT_SECONDS)

def FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_COUNTER, LOCAL_COUNTER, STRING_PREFIX_REGISTER, STRING_PROMPT, STRING_RESPONSE):
    HALF_WIDTH = WIDTH // 2
    STRING_INTRO_FIRST = f' START - {GLOBAL_COUNTER} '.center(WIDTH, '=')
    STRING_INTRO_END = f' END - {GLOBAL_COUNTER} '.center(WIDTH, '=')
    STRING_INTRO = f'''GLOBAL NUMBER: {GLOBAL_COUNTER}
LOCAL NUMBER: {LOCAL_COUNTER}
NAME: {STRING_PREFIX_REGISTER}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    STRING_ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return STRING_INTRO_FIRST + '\n' + STRING_INTRO + STRING_PROMPT + STRING_ENDING + STRING_INTRO_END

def FUNCTION_ATOMIC_WRITE_JSON(FILE_PATH, OBJECT_VALUE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    FILE_TEMP = FILE_PATH.with_suffix(FILE_PATH.suffix + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        json.dump(OBJECT_VALUE, FILE_HANDLE, indent=2, sort_keys=True)
        FILE_HANDLE.write('\n')
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_PATH)

def FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_VALUE):
    if not STRING_VALUE:
        return
    with open(FILE_PATH, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())

def FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PATH, COMMITTED_SIZE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not FILE_PATH.exists():
        if COMMITTED_SIZE != 0:
            raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is missing, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
        FILE_PATH.touch()
        return
    CURRENT_SIZE = FILE_PATH.stat().st_size
    if CURRENT_SIZE < COMMITTED_SIZE:
        raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is only {CURRENT_SIZE} bytes, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
    if CURRENT_SIZE > COMMITTED_SIZE:
        with open(FILE_PATH, 'r+b') as FILE_HANDLE:
            FILE_HANDLE.truncate(COMMITTED_SIZE)
            FILE_HANDLE.flush()
            os.fsync(FILE_HANDLE.fileno())

def FUNCTION_BUILD_WORK_ITEMS():
    STRING_PREFIX_REGISTER_DIGIT = ''
    SPECIAL_NAME_FOR_NUMBERS = ''
    SPECIAL_NAME_FOR_V = False
    with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
        LIST_REGISTER = [EACH_LINE.split(': ', 1)[1].strip() for EACH_LINE in FILE_HANDLE if EACH_LINE.startswith('REGISTER: ')]
    with open(FILE_ALL, encoding='utf-8') as FILE_HANDLE:
        LIST_ALL_LINE = FILE_HANDLE.readlines()
    LIST_FILE_NAME = sorted(os.listdir(FOLDER_INPUT), key=FUNCTION_NATURAL_SORT_KEY)
    LIST_WORK_ITEM = []
    for EACH_REGISTER in LIST_REGISTER:
        LIST_WORD = [EACH_REGISTER]
        if re.search(r'\d', EACH_REGISTER):
            MATCH_PREFIX = re.match(r'^[a-zA-Z]+', EACH_REGISTER)
            if MATCH_PREFIX:
                SPECIAL_NAME_FOR_NUMBERS = MATCH_PREFIX.group(0)
                LIST_WORD.append(SPECIAL_NAME_FOR_NUMBERS)
        elif EACH_REGISTER.endswith('h'):
            STRING_BASE = EACH_REGISTER[:-1]
            if STRING_BASE and STRING_BASE != EACH_REGISTER:
                LIST_WORD.append(STRING_BASE)
        elif EACH_REGISTER.startswith('v'):
            STRING_BASE = EACH_REGISTER[1:]
            if STRING_BASE and STRING_BASE != EACH_REGISTER and any(f'REGISTER: {STRING_BASE}' in EACH_LINE for EACH_LINE in LIST_ALL_LINE):
                LIST_WORD.append(STRING_BASE)
            SPECIAL_NAME_FOR_V = True
        LIST_WORD = list(dict.fromkeys(LIST_WORD))
        LIST_PATH = []
        for EACH_FILE_NAME in LIST_FILE_NAME:
            if not EACH_FILE_NAME.endswith('.txt'):
                continue
            FILE_PATH = os.path.join(FOLDER_INPUT, EACH_FILE_NAME)
            with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
                if any(any(EACH_WORD in EACH_LINE for EACH_WORD in LIST_WORD) for EACH_LINE in FILE_HANDLE):
                    LIST_PATH.append(FILE_PATH)
        LIST_PATH.sort(key=lambda EACH_PATH: int(EACH_PATH.split('SPECIFICATION_')[1].split('.txt')[0]))
        STRING_PREFIX_REGISTER = EACH_REGISTER if STRING_PREFIX_REGISTER_DIGIT == '' else STRING_PREFIX_REGISTER_DIGIT
        if SPECIAL_NAME_FOR_NUMBERS == '':
            SPECIAL_NAME_FOR_NUMBERS_PROMPT = ''
        else:
            SPECIAL_NAME_FOR_NUMBERS_PROMPT = f' or {SPECIAL_NAME_FOR_NUMBERS.strip()}'
        if SPECIAL_NAME_FOR_V:
            SPECIAL_NAME_FOR_NUMBERS_PROMPT = f' or {STRING_PREFIX_REGISTER.strip()[1:]}'
        for LOCAL_COUNTER, EACH_FILE_PATH in enumerate(LIST_PATH, 1):
            LIST_WORK_ITEM.append({'each_register': EACH_REGISTER, 'each_file_path': EACH_FILE_PATH, 'string_prefix_register': STRING_PREFIX_REGISTER, 'special_name_for_numbers_prompt': SPECIAL_NAME_FOR_NUMBERS_PROMPT, 'local_counter': LOCAL_COUNTER})
    return LIST_WORK_ITEM

def FUNCTION_WORK_SHA256(LIST_WORK_ITEM):
    HASH_OBJECT = hashlib.sha256()
    for EACH_INPUT_PATH in [FILE_INPUT, FILE_ALL]:
        with open(EACH_INPUT_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                HASH_OBJECT.update(EACH_CHUNK)
    SET_HASHED_PATH = set()
    for EACH_WORK_ITEM in LIST_WORK_ITEM:
        EACH_FILE_PATH = EACH_WORK_ITEM['each_file_path']
        HASH_OBJECT.update((EACH_WORK_ITEM['each_register'] + '\0' + EACH_FILE_PATH + '\0' + EACH_WORK_ITEM['string_prefix_register'] + '\0' + EACH_WORK_ITEM['special_name_for_numbers_prompt'] + '\0' + str(EACH_WORK_ITEM['local_counter']) + '\n').encode('utf-8'))
        if EACH_FILE_PATH in SET_HASHED_PATH:
            continue
        SET_HASHED_PATH.add(EACH_FILE_PATH)
        with open(EACH_FILE_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                HASH_OBJECT.update(EACH_CHUNK)
    return HASH_OBJECT.hexdigest()

def FUNCTION_CHECKPOINT(NEXT_INDEX):
    global LIST_HISTORY, LIST_OUTPUT_LINE, LOOPS_SINCE_CHECKPOINT
    if LIST_HISTORY:
        STRING_PROMPT_HISTORY = SUFFIX.join(LIST_HISTORY)
        if os.path.getsize(FILE_PROMPT) > 0:
            STRING_PROMPT_HISTORY = SUFFIX + STRING_PROMPT_HISTORY
        FUNCTION_APPEND_AND_SYNC(FILE_PROMPT, STRING_PROMPT_HISTORY)
    if LIST_OUTPUT_LINE:
        STRING_OUTPUT = '\n'.join(LIST_OUTPUT_LINE)
        if os.path.getsize(FILE_OUTPUT) > 0:
            STRING_OUTPUT = '\n' + STRING_OUTPUT
        FUNCTION_APPEND_AND_SYNC(FILE_OUTPUT, STRING_OUTPUT)
    NEW_STATE = {
        'next_index': NEXT_INDEX,
        'next_loop': NEXT_INDEX + 1,
        'global_counter': GLOBAL_COUNTER,
        'work_sha256': WORK_SHA256,
        'paths': {'file_output': FILE_OUTPUT, 'file_prompt': FILE_PROMPT},
        'file_sizes': {'file_output': os.path.getsize(FILE_OUTPUT), 'file_prompt': os.path.getsize(FILE_PROMPT)}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, NEW_STATE)
    print(f'[CHECKPOINT] Saved through work item {NEXT_INDEX} of {len(LIST_WORK_ITEM)}.', flush=True)
    LIST_HISTORY = []
    LIST_OUTPUT_LINE = []
    LOOPS_SINCE_CHECKPOINT = 0
# FUNCTION - FINISH

# VARIABLE - BEGIN
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
LIST_HISTORY = []
LIST_OUTPUT_LINE = []
LOOPS_SINCE_CHECKPOINT = 0
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT, FILE_CHECKPOINT]:
    Path(EACH_FILE_PATH).parent.mkdir(parents=True, exist_ok=True)

LIST_WORK_ITEM = FUNCTION_BUILD_WORK_ITEMS()
WORK_SHA256 = FUNCTION_WORK_SHA256(LIST_WORK_ITEM)

if RESET_PROGRESS and Path(FILE_CHECKPOINT).exists():
    Path(FILE_CHECKPOINT).unlink()

if Path(FILE_CHECKPOINT).exists():
    with open(FILE_CHECKPOINT, encoding='utf-8') as FILE_HANDLE:
        STATE = json.load(FILE_HANDLE)
    if STATE.get('work_sha256') != WORK_SHA256:
        raise RuntimeError('Register/specification inputs changed after the checkpoint was created. Set RESET_PROGRESS = True only if you intentionally want a fresh run.')
    for EACH_PATH_KEY, EACH_FILE_PATH in [('file_output', FILE_OUTPUT), ('file_prompt', FILE_PROMPT)]:
        SAVED_PATH = STATE.get('paths', {}).get(EACH_PATH_KEY)
        if SAVED_PATH != EACH_FILE_PATH:
            raise RuntimeError(f'Checkpoint path mismatch for {EACH_PATH_KEY}: {SAVED_PATH!r} != {EACH_FILE_PATH!r}')
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_OUTPUT, STATE['file_sizes']['file_output'])
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT, STATE['file_sizes']['file_prompt'])
    START_INDEX = STATE['next_index']
    GLOBAL_COUNTER = STATE['global_counter']
    print(f'[RESUME] Continuing from work item {START_INDEX + 1} of {len(LIST_WORK_ITEM)}.', flush=True)
else:
    for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT]:
        with open(EACH_FILE_PATH, 'w', encoding='utf-8'):
            pass
    START_INDEX = 0
    GLOBAL_COUNTER = 1
    STATE = {
        'next_index': 0,
        'next_loop': 1,
        'global_counter': GLOBAL_COUNTER,
        'work_sha256': WORK_SHA256,
        'paths': {'file_output': FILE_OUTPUT, 'file_prompt': FILE_PROMPT},
        'file_sizes': {'file_output': 0, 'file_prompt': 0}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, STATE)

for EACH_INDEX_WORK in range(START_INDEX, len(LIST_WORK_ITEM)):
    EACH_WORK_ITEM = LIST_WORK_ITEM[EACH_INDEX_WORK]
    EACH_REGISTER = EACH_WORK_ITEM['each_register']
    EACH_FILE_PATH = EACH_WORK_ITEM['each_file_path']
    STRING_PREFIX_REGISTER = EACH_WORK_ITEM['string_prefix_register']
    SPECIAL_NAME_FOR_NUMBERS_PROMPT = EACH_WORK_ITEM['special_name_for_numbers_prompt']
    LOCAL_COUNTER = EACH_WORK_ITEM['local_counter']
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        STRING_SPEC = FILE_HANDLE.read()
    STRING_PROMPT = f'''For RISC-V, does this text below contain any information about {STRING_PREFIX_REGISTER}{SPECIAL_NAME_FOR_NUMBERS_PROMPT} register? Output YES or NO only.

The text:

{STRING_SPEC}'''
    STRING_RESPONSE = FUNCTION_CLEAN_TEXT(FUNCTION_LLM_CALL(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT))
    LIST_HISTORY.append(FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_COUNTER, LOCAL_COUNTER, STRING_PREFIX_REGISTER, STRING_PROMPT, STRING_RESPONSE))
    if 'YES' in STRING_RESPONSE:
        LIST_OUTPUT_LINE.append(f'{EACH_REGISTER} - {EACH_FILE_PATH}')
    GLOBAL_COUNTER += 1
    LOOPS_SINCE_CHECKPOINT += 1
    IS_LAST_WORK_ITEM = EACH_INDEX_WORK == len(LIST_WORK_ITEM) - 1
    if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or IS_LAST_WORK_ITEM:
        FUNCTION_CHECKPOINT(EACH_INDEX_WORK + 1)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 3 - CREATE - 3_REGISTER_BOOLEAN.txt - END

49 minute(s) 33.01 second(s)

====
DONE
====


In [ ]:
# 4 - CREATE - 4_REGISTER_RULE.txt - START

"""
Extract architectural behaviors from relevant RISC-V CSR specification sections, transform them into atomic and self-contained rules, and filter out non-actionable rules using LLMs.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client, ResponseError
from pathlib import Path
import hashlib, json, os, re, subprocess
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_PROMPT_1 = '5_REGISTER/2_HISTORY/2_HISTORY_REGISTER_RULE_1.txt'
FILE_PROMPT_2 = '5_REGISTER/2_HISTORY/3_HISTORY_REGISTER_RULE_2.txt'
FILE_CHECKPOINT = '5_REGISTER/3_WHICH_LOOP/WHICH_LOOP_RULE.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'"))

def FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    STRING_ERROR = str(ERROR).lower()
    LIST_RETRY_KEYWORD = ['cuda', 'gpu', 'launch timed out', 'ggml_backend_cuda', 'cudastreamsynchronize', 'cannot access memory', 'runner']
    if STATUS_CODE is not None and STATUS_CODE >= 500:
        return True
    return any(EACH_KEYWORD in STRING_ERROR for EACH_KEYWORD in LIST_RETRY_KEYWORD)

def FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    return STATUS_CODE in (408, 429) or (STATUS_CODE is not None and STATUS_CODE >= 500)

def FUNCTION_STOP_LOCAL_RUNNER(MODEL):
    print(f'[RECOVERY] Stopping Ollama runner for {MODEL!r}...', flush=True)
    try:
        COMPLETED_PROCESS = subprocess.run(['ollama', 'stop', MODEL], check=False, capture_output=True, text=True, timeout=RUNNER_STOP_TIMEOUT_SECONDS)
        if COMPLETED_PROCESS.returncode != 0:
            STRING_ERROR = COMPLETED_PROCESS.stderr.strip()
            if STRING_ERROR:
                print(f'[RECOVERY] ollama stop returned {COMPLETED_PROCESS.returncode}: {STRING_ERROR}', flush=True)
    except Exception as ERROR:
        print(f'[RECOVERY] Could not stop runner cleanly: {ERROR}', flush=True)

def FUNCTION_LLM(STRING_PROMPT):
    NUMBER_ATTEMPT = 0
    while True:
        NUMBER_ATTEMPT += 1
        try:
            if IS_LOCAL:
                return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
            return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
        except (ResponseError, ConnectionError) as ERROR:
            IS_RETRYABLE = FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR) if IS_LOCAL else FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR)
            if not AUTO_RETRY or not IS_RETRYABLE or NUMBER_ATTEMPT >= MAX_RETRIES:
                print(f'[RECOVERY] Giving up after attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}.', flush=True)
                raise
            STRING_MODE = 'LOCAL' if IS_LOCAL else 'ONLINE'
            print(f'[RECOVERY] {STRING_MODE} request failed on attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}: {ERROR}', flush=True)
            if IS_LOCAL:
                FUNCTION_STOP_LOCAL_RUNNER(MODEL)
            print(f'[RECOVERY] Waiting {RETRY_WAIT_SECONDS} seconds, then retrying the SAME request automatically...', flush=True)
            time.sleep(RETRY_WAIT_SECONDS)

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_NORMALIZE_RULE(TEXT_VALUE):
    LIST_LINE = [EACH_LN.strip() for EACH_LN in TEXT_VALUE.splitlines() if EACH_LN.strip()]
    LIST_RULE = []
    for EACH_LN in LIST_LINE:
        if re.match(r'^RULE\s+\d+:', EACH_LN, re.I):
            EACH_LN = re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LN)
            LIST_RULE.append(EACH_LN)
    LIST_FIXED = []
    for EACH_LN in LIST_RULE:
        MATCH_RULE = re.match(r'^\s*RULE\s+(\d+):\s*(.*)', EACH_LN)
        if not MATCH_RULE:
            LIST_FIXED.append(EACH_LN)
            continue
        NUMBER_RULE = MATCH_RULE.group(1)
        STRING_REST = MATCH_RULE.group(2)
        STRING_REST = re.sub(rf'^RULE\s+{NUMBER_RULE}:\s*', '', STRING_REST)
        LIST_FIXED.append(f'RULE {NUMBER_RULE}: {STRING_REST}')
    return '\n'.join(LIST_FIXED)

def FUNCTION_ATOMIC_WRITE_JSON(FILE_PATH, OBJECT_VALUE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    FILE_TEMP = FILE_PATH.with_suffix(FILE_PATH.suffix + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        json.dump(OBJECT_VALUE, FILE_HANDLE, indent=2, sort_keys=True)
        FILE_HANDLE.write('\n')
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_PATH)

def FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_VALUE):
    if not STRING_VALUE:
        return
    with open(FILE_PATH, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())

def FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PATH, COMMITTED_SIZE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not FILE_PATH.exists():
        if COMMITTED_SIZE != 0:
            raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is missing, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
        FILE_PATH.touch()
        return
    CURRENT_SIZE = FILE_PATH.stat().st_size
    if CURRENT_SIZE < COMMITTED_SIZE:
        raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is only {CURRENT_SIZE} bytes, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
    if CURRENT_SIZE > COMMITTED_SIZE:
        with open(FILE_PATH, 'r+b') as FILE_HANDLE:
            FILE_HANDLE.truncate(COMMITTED_SIZE)
            FILE_HANDLE.flush()
            os.fsync(FILE_HANDLE.fileno())

def FUNCTION_WORK_SHA256(LIST_PAIR):
    HASH_OBJECT = hashlib.sha256()
    with open(FILE_INPUT, 'rb') as FILE_HANDLE:
        for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
            HASH_OBJECT.update(EACH_CHUNK)
    SET_HASHED_PATH = set()
    for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
        if EACH_FILE_PATH in SET_HASHED_PATH:
            continue
        SET_HASHED_PATH.add(EACH_FILE_PATH)
        HASH_OBJECT.update(EACH_FILE_PATH.encode('utf-8') + b'\0')
        with open(EACH_FILE_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                HASH_OBJECT.update(EACH_CHUNK)
    return HASH_OBJECT.hexdigest()

def FUNCTION_ATOMIC_WRITE_OUTPUT(STRING_VALUE):
    FILE_TEMP = Path(FILE_OUTPUT + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_OUTPUT)

def FUNCTION_APPEND_HISTORY(FILE_PATH, LIST_HISTORY):
    if not LIST_HISTORY:
        return
    STRING_HISTORY = ''.join(LIST_HISTORY).removesuffix(SUFFIX)
    if os.path.getsize(FILE_PATH) > 0:
        STRING_HISTORY = SUFFIX + STRING_HISTORY
    FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_HISTORY)

def FUNCTION_WRITE_OUTPUT():
    MAP_REGISTER_RULE = defaultdict(list)
    LIST_BLOCK_FINAL = []
    LIST_SUPER_FINAL = []
    for EACH_BLOCK in LIST_BLOCK_OUTPUT:
        LIST_LINE = EACH_BLOCK.strip().splitlines()

        # BONUS FEATURE - DELETE UNWORTHY RULES - BEGIN
        if IS_64_BIT:
            LIST_LINE = [
                EACH_LINE
                for EACH_LINE in LIST_LINE
                if not (
                    PATTERN_RV32.search(EACH_LINE)
                    or PATTERN_32BIT.search(EACH_LINE)
                    or PATTERN_WFI.search(EACH_LINE)
                    or PATTERN_INTERRUPT.search(EACH_LINE)
                    or 'XLEN=32' in EACH_LINE
                    or 'XLEN = 32' in EACH_LINE
                    or 'XLEN is 32' in EACH_LINE
                    or 'XLEN equals 32' in EACH_LINE
                )
            ]
        # BONUS FEATURE - DELETE UNWORTHY RULES - FINISH

        STRING_NAME = LIST_LINE[0].split(':', 1)[1].strip()
        MAP_REGISTER_RULE[STRING_NAME].extend(LIST_LINE[1:])

    for EACH_NAME, EACH_LIST_RULE in MAP_REGISTER_RULE.items():
        LIST_LINE_NEW = [f'REGISTER: {EACH_NAME}']
        for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(EACH_LIST_RULE, 1):
            LIST_LINE_NEW.append(f"RULE {EACH_INDEX_RULE}: {EACH_RULE_LINE.split(':', 1)[1].strip()}")
        LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE_NEW))

    for EACH_BLOCK in LIST_BLOCK_FINAL:
        LIST_LINE = EACH_BLOCK.splitlines()
        STRING_HEADER = LIST_LINE[0].strip()
        LIST_RULE = [EACH_R.strip() for EACH_R in LIST_LINE[1:] if EACH_R.strip() and EACH_R.strip() != 'RULE:']
        LIST_RULE = [re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_R) for EACH_R in LIST_RULE]
        LIST_RULE = [re.sub(r'RULE\s+\d+:', f'RULE {EACH_I}:', EACH_R) for EACH_I, EACH_R in enumerate(LIST_RULE, 1)]
        LIST_SUPER_FINAL.append(STRING_HEADER + '\n' + '\n'.join(LIST_RULE))
    FUNCTION_ATOMIC_WRITE_OUTPUT('\n\n'.join(LIST_SUPER_FINAL))

def FUNCTION_CHECKPOINT(NEXT_INDEX):
    global LIST_HISTORY_1, LIST_HISTORY_2, LOOPS_SINCE_CHECKPOINT
    FUNCTION_APPEND_HISTORY(FILE_PROMPT_1, LIST_HISTORY_1)
    FUNCTION_APPEND_HISTORY(FILE_PROMPT_2, LIST_HISTORY_2)
    NEW_STATE = {
        'next_index': NEXT_INDEX,
        'next_loop': NEXT_INDEX + 1,
        'global_counter_1': GLOBAL_COUNTER_1,
        'local_counter_1': LOCAL_COUNTER_1,
        'global_counter_2': GLOBAL_COUNTER_2,
        'local_counter_2': LOCAL_COUNTER_2,
        'register_initial': REGISTER_INITIAL,
        'work_sha256': WORK_SHA256,
        'list_block_output': LIST_BLOCK_OUTPUT,
        'paths': {'file_output': FILE_OUTPUT, 'file_prompt_1': FILE_PROMPT_1, 'file_prompt_2': FILE_PROMPT_2},
        'file_sizes': {'file_prompt_1': os.path.getsize(FILE_PROMPT_1), 'file_prompt_2': os.path.getsize(FILE_PROMPT_2)}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, NEW_STATE)
    LIST_HISTORY_1 = []
    LIST_HISTORY_2 = []
    LOOPS_SINCE_CHECKPOINT = 0
    FUNCTION_WRITE_OUTPUT()
    print(f'[CHECKPOINT] Saved through input pair {NEXT_INDEX} of {len(LIST_PAIR)}.', flush=True)
# FUNCTION - FINISH

# VARIABLE - BEGIN
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_BLOCK_OUTPUT = []
LIST_HISTORY_1 = []
LIST_HISTORY_2 = []
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
REGISTER_INITIAL = None
IS_64_BIT = True
PATTERN_RV32 = re.compile(r'\bRV32\b')
PATTERN_32BIT = re.compile(r'\b32[\-\u2011]bits?\b')
PATTERN_WFI = re.compile(r'\bWFI\b')
PATTERN_INTERRUPT = re.compile(r'\binterrupts\b', re.IGNORECASE)
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
LOOPS_SINCE_CHECKPOINT = 0
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT_1, FILE_PROMPT_2, FILE_CHECKPOINT]:
    Path(EACH_FILE_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

WORK_SHA256 = FUNCTION_WORK_SHA256(LIST_PAIR)

if RESET_PROGRESS and Path(FILE_CHECKPOINT).exists():
    Path(FILE_CHECKPOINT).unlink()

if Path(FILE_CHECKPOINT).exists():
    with open(FILE_CHECKPOINT, encoding='utf-8') as FILE_HANDLE:
        STATE = json.load(FILE_HANDLE)
    if STATE.get('work_sha256') != WORK_SHA256:
        raise RuntimeError('Register/specification inputs changed. Set RESET_PROGRESS = True only if you want a fresh run.')
    for EACH_PATH_KEY, EACH_FILE_PATH in [('file_output', FILE_OUTPUT), ('file_prompt_1', FILE_PROMPT_1), ('file_prompt_2', FILE_PROMPT_2)]:
        if STATE.get('paths', {}).get(EACH_PATH_KEY) != EACH_FILE_PATH:
            raise RuntimeError(f'Checkpoint path mismatch for {EACH_PATH_KEY}.')
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT_1, STATE['file_sizes']['file_prompt_1'])
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT_2, STATE['file_sizes']['file_prompt_2'])
    LIST_BLOCK_OUTPUT = STATE['list_block_output']
    START_INDEX = STATE['next_index']
    GLOBAL_COUNTER_1 = STATE['global_counter_1']
    LOCAL_COUNTER_1 = STATE['local_counter_1']
    GLOBAL_COUNTER_2 = STATE['global_counter_2']
    LOCAL_COUNTER_2 = STATE['local_counter_2']
    REGISTER_INITIAL = STATE['register_initial']
    FUNCTION_WRITE_OUTPUT()
    print(f'[RESUME] Continuing from input pair {START_INDEX + 1} of {len(LIST_PAIR)}.', flush=True)
else:
    for EACH_FILE_PATH in [FILE_PROMPT_1, FILE_PROMPT_2]:
        with open(EACH_FILE_PATH, 'w', encoding='utf-8'):
            pass
    START_INDEX = 0
    FUNCTION_CHECKPOINT(0)

for EACH_INDEX_PAIR in range(START_INDEX, len(LIST_PAIR)):
    EACH_REGISTER, EACH_FILE_PATH = LIST_PAIR[EACH_INDEX_PAIR]
    if EACH_INDEX_PAIR == 0:
        REGISTER_INITIAL = EACH_REGISTER
    elif EACH_REGISTER != REGISTER_INITIAL:
        REGISTER_INITIAL = EACH_REGISTER
        LOCAL_COUNTER_1 = 1
        LOCAL_COUNTER_2 = 1
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        STRING_SPEC = FILE_HANDLE.read()

    # PROMPT 1 - BEGIN
    STRING_PROMPT_1 = f'''For RISC-V, please extract all possible information about {EACH_REGISTER} register from the text below. The information must focus on the {EACH_REGISTER} register.

However, you MUST include any other registers, fields, or bits
IF AND ONLY IF they are the direct cause of {EACH_REGISTER} behavior.

Each extracted item MUST state exactly what the source text says about {EACH_REGISTER}. If the source gives an explicit condition or triggering context, include it. If the source does not give an explicit condition, do not invent one.

Extract ONLY explicit, testable, normative statements, and explicit descriptive statements about architectural behavior, from the provided text.

Please output in paragraphs. If you want, you can include bullet or numbered points in your output. You can also include tables. However, do not give me any introduction in your output. You just need to output the information that you extract.

If your output includes some sentences that refer to some tables or figures, include those tables or figures too.

Remember, output must be ASCII-only text.

If it happens to be there is no information about {EACH_REGISTER} register from the text below, output NONE

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Do NOT infer the opposite case unless the source explicitly states it. In particular, do NOT invent a rule for "when something does not happen" unless that case is explicitly stated in the source text.

Do NOT expand a brief relationship such as "is mirrored in" into stronger claims such as bidirectional write propagation, synchronization behavior, or update rules unless the source explicitly states them.

Do not combine or rewrite conditions. Each condition must exactly match the logical structure in the source text, and must not introduce contradictions.

Each extracted item MUST include both the condition and its architectural effect (e.g., what exception is raised or what behavior occurs).

If the text includes tables, each row of the table MUST be treated as a separate rule and fully extracted.

Your description must be very clear and unambiguous. If you want, you can output in bullet points also.

Use the exact terminology and wording used in the text below as closely as possible; do not substitute synonyms, paraphrase architectural terms, or strengthen/weaken the stated behavior.

PLEASE preserve instruction-specific scope exactly. If the source states a rule for a specific instruction, include the instruction name in the extracted rule and do not turn it into a generic register rule.

The text below:

{STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_LLM(STRING_PROMPT_1)
    if STRING_RESPONSE_1.strip() == 'NONE':
        LOOPS_SINCE_CHECKPOINT += 1
        if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or EACH_INDEX_PAIR == len(LIST_PAIR) - 1:
            FUNCTION_CHECKPOINT(EACH_INDEX_PAIR + 1)
        continue
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(STRING_RESPONSE_1)
    LIST_HISTORY_1.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_1, LOCAL_COUNTER_1, EACH_REGISTER, STRING_PROMPT_1, STRING_RESPONSE_1))
    # PROMPT 1 - FINISH

    # PROMPT 2 - BEGIN
    STRING_PROMPT_2 = f'''For RISC-V, for {EACH_REGISTER} register, rewrite the following description as a list of rules. Each rule must explain about {EACH_REGISTER} register. Essentially, for each rule, it is about {EACH_REGISTER} register, not other things.

Do not lose any information about {EACH_REGISTER} register from the description. Read the text or the description carefully. Understand the context and the sentences. Do not lose any important information.

For each rule, do not use any internal references like "According to Table 7," or "see Figure 3." Each rule must be one sentence only. Each rule must stand fully on its own without cross-references. Do not quote or refer to other rules. Essentially, each rule is one idea. Avoid a rule like this: "The register being described is {EACH_REGISTER}."

Do NOT refer to other rules by number, name, or description. Phrases such as "described in RULE N", "in the previous rule", "the rule above", "as stated earlier", or "the same condition as RULE N" are FORBIDDEN. If two rules share context or conditions, restate the shared context fully in each rule rather than referencing the other rule. Each rule must be fully self-contained and understandable in isolation, even if read out of order or extracted individually.

Each rule should become a single faithful statement derived directly from the source text. A rule may be normative or descriptive, as long as it states explicit architectural information about {EACH_REGISTER} from the source text.

Each rule should be a separate atomic rule that expresses one explicit architectural statement about {EACH_REGISTER} from the source text.

If a sentence contains multiple independent events or clearly different behaviors, split them into separate rules so that each rule expresses one case.

However, if a sentence describes a single event that can legitimately produce multiple alternative outcomes (e.g., "A may cause X or Y"), do NOT split the rule. In such cases, keep the alternatives in a single rule, because the specification is describing implementation-dependent outcomes where either result is acceptable.

Each rule needs to be clear and detailed. Extract ONLY explicit, testable, normative statements, and explicit descriptive architectural statements, about {EACH_REGISTER}. Do not introduce stronger behavioral semantics than the source text states.

Remember, if a sentence contains multiple distinct behaviors or enumerated cases, split them into multiple rules so each rule expresses only one case.

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Do NOT infer the opposite case unless the source explicitly states it. In particular, do NOT invent a rule for when something does not happen unless that case is explicitly stated in the source text.

Do NOT convert descriptive wording into a stronger operational rule unless the source explicitly states that stronger rule. For example, do not convert "indicates" into "is set to 1" or "is set to 0" unless the source explicitly says so.

Do NOT expand a brief relationship such as "is mirrored in" into stronger claims such as bidirectional write propagation or update rules unless the source explicitly states them.

If the source text states a descriptive property of {EACH_REGISTER} that is explicit and architecturally relevant, preserve it as a rule even if it is not written as an event -> outcome statement.

It is acceptable to include rules about extension dependencies, optional presence conditions, access restrictions, and field semantics such as WARL behavior, as long as they are explicit, normative, and testable architectural rules about {EACH_REGISTER}.

Do NOT remove or omit statements that explicitly classify fields as WARL; such classifications must be preserved as part of the rule, even if they appear descriptive.

Exclude any rule that contains phrases such as "implementations may/should provide" or "may choose to implement", even if the sentence also contains behavioral details.

OUTPUT FORMAT:
+ Each rule must begin with the word "RULE", followed by its number (e.g., "RULE 1: ", "RULE 2: ", "RULE 3: ", ...).
+ Do not give me any introduction.
+ Each rule must contain the word {EACH_REGISTER}.

Remember, output must be ASCII-only text.

The following description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_2))
    LIST_HISTORY_2.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_2, LOCAL_COUNTER_2, EACH_REGISTER, STRING_PROMPT_2, STRING_RESPONSE_2))
    # PROMPT 2 - FINISH

    # PROMPT 3 - START
    STRING_PROMPT_3 = f'''Given the following list of rules about a RISC-V CSR, remove any rule that is not actionable. A rule is actionable if it describes a concrete observable behavior that can be verified by executing instructions on a RISC-V implementation, such as: the value read back from the CSR, whether an exception is raised or not raised, which privilege modes can access the CSR, how the CSR value changes after specific operations, or whether specific field values are reserved, invalid, or constrained.

Keep rules about reserved, invalid, or constrained CSR field values when they identify specific values or value ranges for a concrete field, because these can be tested by writing those values and observing readback behavior, trapping behavior, or other architectural effects.

Remove any rule that is purely identifying, such as CSR address encoding, count of CSRs in an extension, or abstract architectural purpose.

Do not rewrite, rephrase, or add to any rule. Output only the surviving rules exactly as they appeared in the input, renumbered starting from 1.

Each output rule must begin with "RULE" followed by its number. Output must be ASCII-only text. Do not give any introduction.

{STRING_RESPONSE_2}'''
    STRING_RESPONSE_3 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_3))
    # PROMPT 3 - FINISH

    # FINALIZING - BEGIN
    STRING_RESPONSE_FINISH = STRING_RESPONSE_3
    STRING_CLEANED = FUNCTION_NORMALIZE_RULE(STRING_RESPONSE_FINISH).replace('\u2011', '-')
    LIST_BLOCK_OUTPUT.append(f'REGISTER: {EACH_REGISTER}\n{STRING_CLEANED}')
    # FINALIZING - FINISH

    # CLEANING UP PROMPT - BEGIN
    GLOBAL_COUNTER_1 += 1
    LOCAL_COUNTER_1 += 1
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1
    # CLEANING UP PROMPT - FINISH

    # CHECKPOINT - BEGIN
    LOOPS_SINCE_CHECKPOINT += 1
    if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or EACH_INDEX_PAIR == len(LIST_PAIR) - 1:
        FUNCTION_CHECKPOINT(EACH_INDEX_PAIR + 1)
    # CHECKPOINT - FINISH
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 4 - CREATE - 4_REGISTER_RULE.txt - END

8 hour(s) 1 minute(s) 7.91 second(s)

====
DONE
====


In [ ]:
# 5 - CREATE - 5_REGISTER_EXTENSION.txt - START

"""
Identify and extract RISC-V extensions associated with each selected CSR from relevant specification sections using LLMs, then normalize and deduplicate the extracted extension names.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client, ResponseError
from pathlib import Path
import hashlib, json, os, re, subprocess
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_PROMPT = '5_REGISTER/2_HISTORY/4_HISTORY_REGISTER_EXTENSION.txt'
FILE_CHECKPOINT = '5_REGISTER/3_WHICH_LOOP/WHICH_LOOP_EXTENSION.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    STRING_ERROR = str(ERROR).lower()
    LIST_RETRY_KEYWORD = ['cuda', 'gpu', 'launch timed out', 'ggml_backend_cuda', 'cudastreamsynchronize', 'cannot access memory', 'runner']
    if STATUS_CODE is not None and STATUS_CODE >= 500:
        return True
    return any(EACH_KEYWORD in STRING_ERROR for EACH_KEYWORD in LIST_RETRY_KEYWORD)

def FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    return STATUS_CODE in (408, 429) or (STATUS_CODE is not None and STATUS_CODE >= 500)

def FUNCTION_STOP_LOCAL_RUNNER(MODEL):
    print(f'[RECOVERY] Stopping Ollama runner for {MODEL!r}...', flush=True)
    try:
        COMPLETED_PROCESS = subprocess.run(['ollama', 'stop', MODEL], check=False, capture_output=True, text=True, timeout=RUNNER_STOP_TIMEOUT_SECONDS)
        if COMPLETED_PROCESS.returncode != 0:
            STRING_ERROR = COMPLETED_PROCESS.stderr.strip()
            if STRING_ERROR:
                print(f'[RECOVERY] ollama stop returned {COMPLETED_PROCESS.returncode}: {STRING_ERROR}', flush=True)
    except Exception as ERROR:
        print(f'[RECOVERY] Could not stop runner cleanly: {ERROR}', flush=True)

def FUNCTION_LLM(STRING_PROMPT):
    NUMBER_ATTEMPT = 0
    while True:
        NUMBER_ATTEMPT += 1
        try:
            if IS_LOCAL:
                return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
            return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
        except (ResponseError, ConnectionError) as ERROR:
            IS_RETRYABLE = FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR) if IS_LOCAL else FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR)
            if not AUTO_RETRY or not IS_RETRYABLE or NUMBER_ATTEMPT >= MAX_RETRIES:
                print(f'[RECOVERY] Giving up after attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}.', flush=True)
                raise
            STRING_MODE = 'LOCAL' if IS_LOCAL else 'ONLINE'
            print(f'[RECOVERY] {STRING_MODE} request failed on attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}: {ERROR}', flush=True)
            if IS_LOCAL:
                FUNCTION_STOP_LOCAL_RUNNER(MODEL)
            print(f'[RECOVERY] Waiting {RETRY_WAIT_SECONDS} seconds, then retrying the SAME request automatically...', flush=True)
            time.sleep(RETRY_WAIT_SECONDS)

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'"))

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_SPLIT_BLOCK(TEXT_VALUE: str):
    LIST_BLOCK = []
    LIST_CURRENT = []
    for EACH_LINE in TEXT_VALUE.splitlines(True):
        if EACH_LINE.startswith('REGISTER:') and LIST_CURRENT:
            LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
            LIST_CURRENT = []
        LIST_CURRENT.append(EACH_LINE)
    if LIST_CURRENT:
        LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
    return LIST_BLOCK

def FUNCTION_TOKEN_OK(STRING_TOKEN: str):
    STRING_TOKEN = STRING_TOKEN.strip()
    if not STRING_TOKEN or ' ' in STRING_TOKEN or not STRING_TOKEN.isalpha():
        return False
    return STRING_TOKEN[0].isupper() and (len(STRING_TOKEN) == 1 or STRING_TOKEN[1:].islower())

def FUNCTION_ATOMIC_WRITE_JSON(FILE_PATH, OBJECT_VALUE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    FILE_TEMP = FILE_PATH.with_suffix(FILE_PATH.suffix + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        json.dump(OBJECT_VALUE, FILE_HANDLE, indent=2)
        FILE_HANDLE.write('\n')
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_PATH)

def FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_VALUE):
    if not STRING_VALUE:
        return
    with open(FILE_PATH, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())

def FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PATH, COMMITTED_SIZE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not FILE_PATH.exists():
        if COMMITTED_SIZE != 0:
            raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is missing, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
        FILE_PATH.touch()
        return
    CURRENT_SIZE = FILE_PATH.stat().st_size
    if CURRENT_SIZE < COMMITTED_SIZE:
        raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is only {CURRENT_SIZE} bytes, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
    if CURRENT_SIZE > COMMITTED_SIZE:
        with open(FILE_PATH, 'r+b') as FILE_HANDLE:
            FILE_HANDLE.truncate(COMMITTED_SIZE)
            FILE_HANDLE.flush()
            os.fsync(FILE_HANDLE.fileno())

def FUNCTION_WORK_SHA256(LIST_PAIR):
    HASH_OBJECT = hashlib.sha256()
    with open(FILE_INPUT, 'rb') as FILE_HANDLE:
        for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
            HASH_OBJECT.update(EACH_CHUNK)
    SET_HASHED_PATH = set()
    for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
        HASH_OBJECT.update((EACH_REGISTER + '\0' + EACH_FILE_PATH + '\n').encode('utf-8'))
        if EACH_FILE_PATH in SET_HASHED_PATH:
            continue
        SET_HASHED_PATH.add(EACH_FILE_PATH)
        with open(EACH_FILE_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                HASH_OBJECT.update(EACH_CHUNK)
    return HASH_OBJECT.hexdigest()

def FUNCTION_ATOMIC_WRITE_OUTPUT(STRING_VALUE):
    FILE_TEMP = Path(FILE_OUTPUT + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_OUTPUT)

def FUNCTION_CHECKPOINT(NEXT_INDEX):
    global LIST_HISTORY, LOOPS_SINCE_CHECKPOINT
    if LIST_HISTORY:
        STRING_PROMPT_HISTORY = ''.join(LIST_HISTORY).removesuffix(SUFFIX)
        if os.path.getsize(FILE_PROMPT) > 0:
            STRING_PROMPT_HISTORY = SUFFIX + STRING_PROMPT_HISTORY
        FUNCTION_APPEND_AND_SYNC(FILE_PROMPT, STRING_PROMPT_HISTORY)
    NEW_STATE = {
        'next_index': NEXT_INDEX,
        'next_loop': NEXT_INDEX + 1,
        'global_counter': GLOBAL_COUNTER,
        'map_merged': MAP_MERGED,
        'work_sha256': WORK_SHA256,
        'paths': {'file_output': FILE_OUTPUT, 'file_prompt': FILE_PROMPT},
        'file_sizes': {'file_prompt': os.path.getsize(FILE_PROMPT)}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, NEW_STATE)
    LIST_HISTORY = []
    LOOPS_SINCE_CHECKPOINT = 0
    print(f'[CHECKPOINT] Saved through register {NEXT_INDEX} of {len(LIST_REGISTER_ORDER)}.', flush=True)
# FUNCTION - FINISH

# VARIABLE - BEGIN
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_HISTORY = []
GLOBAL_COUNTER = 1
LOCAL_COUNTER = 1
REGISTER_INITIAL = None
MAP_MERGED = {}
LIST_OUTPUT_ENTRY = []
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
LOOPS_SINCE_CHECKPOINT = 0
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT, FILE_CHECKPOINT]:
    Path(EACH_FILE_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

WORK_SHA256 = FUNCTION_WORK_SHA256(LIST_PAIR)

if RESET_PROGRESS and Path(FILE_CHECKPOINT).exists():
    Path(FILE_CHECKPOINT).unlink()

if Path(FILE_CHECKPOINT).exists():
    with open(FILE_CHECKPOINT, encoding='utf-8') as FILE_HANDLE:
        STATE = json.load(FILE_HANDLE)
    if STATE.get('work_sha256') != WORK_SHA256:
        raise RuntimeError('Register/specification inputs changed. Set RESET_PROGRESS = True only if you want a fresh run.')
    for EACH_PATH_KEY, EACH_FILE_PATH in [('file_output', FILE_OUTPUT), ('file_prompt', FILE_PROMPT)]:
        SAVED_PATH = STATE.get('paths', {}).get(EACH_PATH_KEY)
        if SAVED_PATH != EACH_FILE_PATH:
            raise RuntimeError(f'Checkpoint path mismatch for {EACH_PATH_KEY}: {SAVED_PATH!r} != {EACH_FILE_PATH!r}')
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT, STATE['file_sizes']['file_prompt'])
    START_INDEX = STATE['next_index']
    GLOBAL_COUNTER = STATE['global_counter']
    MAP_MERGED = dict(STATE['map_merged'])
    print(f'[RESUME] Continuing from register {START_INDEX + 1} of {len(LIST_REGISTER_ORDER)}.', flush=True)
else:
    for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT]:
        with open(EACH_FILE_PATH, 'w', encoding='utf-8'):
            pass
    START_INDEX = 0
    FUNCTION_CHECKPOINT(0)

for EACH_INDEX_REGISTER in range(START_INDEX, len(LIST_REGISTER_ORDER)):
    EACH_REGISTER = LIST_REGISTER_ORDER[EACH_INDEX_REGISTER]
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER]).strip()
    LOCAL_COUNTER = 1
    STRING_PROMPT = f'''For RISC-V, based on the text below, output all extensions (RISC-V extensions) for {EACH_REGISTER} register. Note: registers, CSRs, fields, modes, interrupts, and instructions are not extensions.

Read the text below carefully and make sure you don't miss any extensions!

HOW TO OUTPUT:
Enter extensions as a comma-separated list on one line. If there are no extensions for {EACH_REGISTER} register in the rules, output NONE.

The text below:

{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT))
    LIST_HISTORY.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER, LOCAL_COUNTER, EACH_REGISTER, STRING_PROMPT, STRING_RESPONSE))
    STRING_RESPONSE_2 = STRING_RESPONSE.replace(EACH_REGISTER, '')
    if 'NONE' not in STRING_RESPONSE_2:
        if EACH_REGISTER in MAP_MERGED:
            MAP_MERGED[EACH_REGISTER] = f'{MAP_MERGED[EACH_REGISTER]}, {STRING_RESPONSE_2}'
        else:
            MAP_MERGED[EACH_REGISTER] = STRING_RESPONSE_2
    GLOBAL_COUNTER += 1
    LOCAL_COUNTER += 1
    LOOPS_SINCE_CHECKPOINT += 1
    IS_LAST_REGISTER = EACH_INDEX_REGISTER == len(LIST_REGISTER_ORDER) - 1
    if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or IS_LAST_REGISTER:
        FUNCTION_CHECKPOINT(EACH_INDEX_REGISTER + 1)

for EACH_REGISTER in sorted(MAP_MERGED, key=str.lower):
    SET_SEEN = set()
    LIST_TOKEN = []
    for EACH_TOKEN in MAP_MERGED[EACH_REGISTER].split(','):
        STRING_TOKEN = EACH_TOKEN.strip()
        if STRING_TOKEN and STRING_TOKEN not in SET_SEEN:
            SET_SEEN.add(STRING_TOKEN)
            LIST_TOKEN.append(STRING_TOKEN)
    LIST_TOKEN = [EACH_TOKEN for EACH_TOKEN in LIST_TOKEN if FUNCTION_TOKEN_OK(EACH_TOKEN)]
    LIST_OUTPUT_ENTRY.append(f"REGISTER: {EACH_REGISTER}\nEXTENSIONS: {', '.join(LIST_TOKEN)}")

FUNCTION_ATOMIC_WRITE_OUTPUT('\n\n'.join(LIST_OUTPUT_ENTRY))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 5 - CREATE - 5_REGISTER_EXTENSION.txt - END

26 minute(s) 3.85 second(s)

====
DONE
====


In [ ]:
# 6 - CREATE - 6_REGISTER_FIELD.txt - START

"""
Extract missing RISC-V CSR bit fields from relevant specification sections using LLMs and normalize them into a structured format containing field names and bit ranges.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client, ResponseError
from pathlib import Path
import hashlib, json, os, re, subprocess
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_PROMPT_1 = '5_REGISTER/2_HISTORY/5_HISTORY_REGISTER_FIELD_1.txt'
FILE_PROMPT_2 = '5_REGISTER/2_HISTORY/6_HISTORY_REGISTER_FIELD_2.txt'
FILE_OTHER_WIDTH = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_OTHER_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_CHECKPOINT = '5_REGISTER/3_WHICH_LOOP/WHICH_LOOP_FIELD.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    STRING_ERROR = str(ERROR).lower()
    LIST_RETRY_KEYWORD = ['cuda', 'gpu', 'launch timed out', 'ggml_backend_cuda', 'cudastreamsynchronize', 'cannot access memory', 'runner']
    if STATUS_CODE is not None and STATUS_CODE >= 500:
        return True
    return any(EACH_KEYWORD in STRING_ERROR for EACH_KEYWORD in LIST_RETRY_KEYWORD)

def FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    return STATUS_CODE in (408, 429) or (STATUS_CODE is not None and STATUS_CODE >= 500)

def FUNCTION_STOP_LOCAL_RUNNER(MODEL):
    print(f'[RECOVERY] Stopping Ollama runner for {MODEL!r}...', flush=True)
    try:
        COMPLETED_PROCESS = subprocess.run(['ollama', 'stop', MODEL], check=False, capture_output=True, text=True, timeout=RUNNER_STOP_TIMEOUT_SECONDS)
        if COMPLETED_PROCESS.returncode != 0:
            STRING_ERROR = COMPLETED_PROCESS.stderr.strip()
            if STRING_ERROR:
                print(f'[RECOVERY] ollama stop returned {COMPLETED_PROCESS.returncode}: {STRING_ERROR}', flush=True)
    except Exception as ERROR:
        print(f'[RECOVERY] Could not stop runner cleanly: {ERROR}', flush=True)

def FUNCTION_LLM(STRING_PROMPT):
    NUMBER_ATTEMPT = 0
    while True:
        NUMBER_ATTEMPT += 1
        try:
            if IS_LOCAL:
                return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
            return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
        except (ResponseError, ConnectionError) as ERROR:
            IS_RETRYABLE = FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR) if IS_LOCAL else FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR)
            if not AUTO_RETRY or not IS_RETRYABLE or NUMBER_ATTEMPT >= MAX_RETRIES:
                print(f'[RECOVERY] Giving up after attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}.', flush=True)
                raise
            STRING_MODE = 'LOCAL' if IS_LOCAL else 'ONLINE'
            print(f'[RECOVERY] {STRING_MODE} request failed on attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}: {ERROR}', flush=True)
            if IS_LOCAL:
                FUNCTION_STOP_LOCAL_RUNNER(MODEL)
            print(f'[RECOVERY] Waiting {RETRY_WAIT_SECONDS} seconds, then retrying the SAME request automatically...', flush=True)
            time.sleep(RETRY_WAIT_SECONDS)

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'"))

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_SPLIT_BLOCK(LIST_LINE):
    LIST_BLOCK = []
    LIST_CURRENT = []
    for EACH_LINE in LIST_LINE:
        if EACH_LINE.startswith('REGISTER:') and LIST_CURRENT:
            LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
            LIST_CURRENT = []
        LIST_CURRENT.append(EACH_LINE)
    if LIST_CURRENT:
        LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
    return LIST_BLOCK

def FUNCTION_ATOMIC_WRITE_JSON(FILE_PATH, OBJECT_VALUE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    FILE_TEMP = FILE_PATH.with_suffix(FILE_PATH.suffix + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        json.dump(OBJECT_VALUE, FILE_HANDLE, indent=2)
        FILE_HANDLE.write('\n')
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_PATH)

def FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_VALUE):
    if not STRING_VALUE:
        return
    with open(FILE_PATH, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())

def FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PATH, COMMITTED_SIZE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not FILE_PATH.exists():
        if COMMITTED_SIZE != 0:
            raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is missing, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
        FILE_PATH.touch()
        return
    CURRENT_SIZE = FILE_PATH.stat().st_size
    if CURRENT_SIZE < COMMITTED_SIZE:
        raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is only {CURRENT_SIZE} bytes, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
    if CURRENT_SIZE > COMMITTED_SIZE:
        with open(FILE_PATH, 'r+b') as FILE_HANDLE:
            FILE_HANDLE.truncate(COMMITTED_SIZE)
            FILE_HANDLE.flush()
            os.fsync(FILE_HANDLE.fileno())

def FUNCTION_WORK_SHA256(LIST_PAIR):
    HASH_OBJECT = hashlib.sha256()
    for EACH_INPUT_PATH in [FILE_INPUT, FILE_OTHER_WIDTH, FILE_OTHER_INPUT]:
        with open(EACH_INPUT_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                HASH_OBJECT.update(EACH_CHUNK)
    SET_HASHED_PATH = set()
    for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
        HASH_OBJECT.update((EACH_REGISTER + '\0' + EACH_FILE_PATH + '\n').encode('utf-8'))
        if EACH_FILE_PATH in SET_HASHED_PATH:
            continue
        SET_HASHED_PATH.add(EACH_FILE_PATH)
        with open(EACH_FILE_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                HASH_OBJECT.update(EACH_CHUNK)
    return HASH_OBJECT.hexdigest()

def FUNCTION_APPEND_HISTORY(FILE_PATH, LIST_HISTORY):
    if not LIST_HISTORY:
        return
    STRING_HISTORY = ''.join(LIST_HISTORY).removesuffix(SUFFIX)
    if os.path.getsize(FILE_PATH) > 0:
        STRING_HISTORY = SUFFIX + STRING_HISTORY
    FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_HISTORY)

def FUNCTION_CHECKPOINT(NEXT_INDEX):
    global LIST_HISTORY_1, LIST_HISTORY_2, LIST_OUTPUT_BLOCK, LOOPS_SINCE_CHECKPOINT
    FUNCTION_APPEND_HISTORY(FILE_PROMPT_1, LIST_HISTORY_1)
    FUNCTION_APPEND_HISTORY(FILE_PROMPT_2, LIST_HISTORY_2)
    if LIST_OUTPUT_BLOCK:
        STRING_OUTPUT = '\n\n'.join(LIST_OUTPUT_BLOCK)
        if os.path.getsize(FILE_OUTPUT) > 0:
            STRING_OUTPUT = '\n\n' + STRING_OUTPUT
        FUNCTION_APPEND_AND_SYNC(FILE_OUTPUT, STRING_OUTPUT)
    NEW_STATE = {
        'next_index': NEXT_INDEX,
        'next_loop': NEXT_INDEX + 1,
        'global_counter_1': GLOBAL_COUNTER_1,
        'local_counter_1': LOCAL_COUNTER_1,
        'global_counter_2': GLOBAL_COUNTER_2,
        'local_counter_2': LOCAL_COUNTER_2,
        'register_initial': REGISTER_INITIAL,
        'work_sha256': WORK_SHA256,
        'paths': {'file_output': FILE_OUTPUT, 'file_prompt_1': FILE_PROMPT_1, 'file_prompt_2': FILE_PROMPT_2},
        'file_sizes': {'file_output': os.path.getsize(FILE_OUTPUT), 'file_prompt_1': os.path.getsize(FILE_PROMPT_1), 'file_prompt_2': os.path.getsize(FILE_PROMPT_2)}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, NEW_STATE)
    LIST_HISTORY_1 = []
    LIST_HISTORY_2 = []
    LIST_OUTPUT_BLOCK = []
    LOOPS_SINCE_CHECKPOINT = 0
    print(f'[CHECKPOINT] Saved through register {NEXT_INDEX} of {len(LIST_REGISTER_NAME)}.', flush=True)
# FUNCTION - FINISH

# VARIABLE - BEGIN
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_REGISTER_NAME = []
MAP_WIDTH = {}
LIST_INJECTED = []
LIST_OUTPUT_BLOCK = []
LIST_HISTORY_1 = []
LIST_HISTORY_2 = []
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
REGISTER_INITIAL = None
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
LOOPS_SINCE_CHECKPOINT = 0
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT_1, FILE_PROMPT_2, FILE_CHECKPOINT]:
    Path(EACH_FILE_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(FILE_OTHER_WIDTH, encoding='utf-8') as FILE_HANDLE:
    STRING_REGISTER = None
    STRING_WIDTH = None
    for EACH_LINE in FILE_HANDLE:
        if EACH_LINE.startswith('REGISTER: '):
            STRING_REGISTER = EACH_LINE.split(': ', 1)[1].strip()
        elif EACH_LINE.startswith('WIDTH: '):
            STRING_WIDTH = EACH_LINE.split(': ', 1)[1].strip()
        elif (EACH_LINE.startswith('FIELDS: -') and STRING_REGISTER is not None and STRING_WIDTH is not None):
            STRING_PAIR = f'{STRING_REGISTER} - {STRING_WIDTH}'
            LIST_REGISTER_NAME.append(STRING_PAIR)
            MAP_WIDTH[STRING_REGISTER] = STRING_WIDTH

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_RULE_LINE = FILE_HANDLE.readlines()

for EACH_LINE in LIST_RULE_LINE:
    if EACH_LINE.startswith('REGISTER:'):
        LIST_INJECTED.append(EACH_LINE)
        STRING_REGISTER = EACH_LINE.split('REGISTER:', 1)[1].strip()
        if STRING_REGISTER in MAP_WIDTH:
            LIST_INJECTED.append(f'WIDTH: {MAP_WIDTH[STRING_REGISTER]}\n')

LIST_INPUT_2 = FUNCTION_SPLIT_BLOCK(LIST_INJECTED)
MAP_BLOCK_BY_REGISTER = {EACH_BLOCK.splitlines()[0].split('REGISTER:', 1)[1].strip(): EACH_BLOCK for EACH_BLOCK in LIST_INPUT_2}

with open(FILE_OTHER_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

WORK_SHA256 = FUNCTION_WORK_SHA256(LIST_PAIR)

if RESET_PROGRESS and Path(FILE_CHECKPOINT).exists():
    Path(FILE_CHECKPOINT).unlink()

if Path(FILE_CHECKPOINT).exists():
    with open(FILE_CHECKPOINT, encoding='utf-8') as FILE_HANDLE:
        STATE = json.load(FILE_HANDLE)
    if STATE.get('work_sha256') != WORK_SHA256:
        raise RuntimeError('Register/rule/specification inputs changed. Set RESET_PROGRESS = True only if you want a fresh run.')
    for EACH_PATH_KEY, EACH_FILE_PATH in [('file_output', FILE_OUTPUT), ('file_prompt_1', FILE_PROMPT_1), ('file_prompt_2', FILE_PROMPT_2)]:
        SAVED_PATH = STATE.get('paths', {}).get(EACH_PATH_KEY)
        if SAVED_PATH != EACH_FILE_PATH:
            raise RuntimeError(f'Checkpoint path mismatch for {EACH_PATH_KEY}: {SAVED_PATH!r} != {EACH_FILE_PATH!r}')
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_OUTPUT, STATE['file_sizes']['file_output'])
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT_1, STATE['file_sizes']['file_prompt_1'])
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT_2, STATE['file_sizes']['file_prompt_2'])
    START_INDEX = STATE['next_index']
    GLOBAL_COUNTER_1 = STATE['global_counter_1']
    LOCAL_COUNTER_1 = STATE['local_counter_1']
    GLOBAL_COUNTER_2 = STATE['global_counter_2']
    LOCAL_COUNTER_2 = STATE['local_counter_2']
    REGISTER_INITIAL = STATE['register_initial']
    print(f'[RESUME] Continuing from register {START_INDEX + 1} of {len(LIST_REGISTER_NAME)}.', flush=True)
else:
    for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT_1, FILE_PROMPT_2]:
        with open(EACH_FILE_PATH, 'w', encoding='utf-8'):
            pass
    START_INDEX = 0
    FUNCTION_CHECKPOINT(0)

for EACH_INDEX_SPECIAL in range(START_INDEX, len(LIST_REGISTER_NAME)):
    EACH_SPECIAL = LIST_REGISTER_NAME[EACH_INDEX_SPECIAL]
    STRING_REGISTER, STRING_WIDTH = (EACH_PART.strip() for EACH_PART in EACH_SPECIAL.split(' - ', 1))
    STRING_BLOCK = MAP_BLOCK_BY_REGISTER.get(STRING_REGISTER, '')
    if EACH_INDEX_SPECIAL == 0:
        REGISTER_INITIAL = STRING_REGISTER
    elif STRING_REGISTER != REGISTER_INITIAL:
        REGISTER_INITIAL = STRING_REGISTER
        LOCAL_COUNTER_1 = 1
        LOCAL_COUNTER_2 = 1
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[STRING_REGISTER]).strip()
    STRING_PROMPT_1 = f'''For RISC-V, based on the text below, output all fields for {STRING_REGISTER} register. Do not give any introduction.

The text below:

{STRING_BLOCK}
TEXT:
{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_1))
    LIST_HISTORY_1.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_1, LOCAL_COUNTER_1, STRING_REGISTER, STRING_PROMPT_1, STRING_RESPONSE_1))
    NUMBER_WIDTH = int(STRING_WIDTH) if STRING_WIDTH.isdigit() else 64
    STRING_PROMPT_2 = f'''For RISC-V, from the description below, extract the fields for {STRING_REGISTER} register. The register of {STRING_REGISTER} is a {NUMBER_WIDTH}-bit register.

OUTPUT FORMAT:
FIELDS: <high_bit>..<low_bit>=<FIELD_NAME>, <high_bit>..<low_bit>=<FIELD_NAME>, ...

RULES:
+ Range Delimiter -> Always use two dots (..) for inclusive ranges (e.g., 7..0). Repeat the index for single-bit fields (e.g., 3..3).
+ Coverage -> Define every bit from {NUMBER_WIDTH - 1} down to 0, in contiguous, non-overlapping ranges. No gaps.
+ Reserved Bits -> If the description says something like "unused", "future use", or "reserved", then the <FIELD_NAME> must be WPRI.
+ Unknown Positions -> If a field's bit range is not given in the description, append the field at the end. The <high_bit> and the <low_bit> become "UNKNOWN". For example, "UNKNOWN..UNKNOWN=<FIELD_NAME>".
+ Ordering -> Based on the position or the bit range, list fields strictly from high to low, separated by ", " (comma and space).
+ No Overlaps -> Ensure bit ranges don't overlap.

Do not output any introduction.

The description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_2))
    LIST_HISTORY_2.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_2, LOCAL_COUNTER_2, STRING_REGISTER, STRING_PROMPT_2, STRING_RESPONSE_2))
    LIST_OUTPUT_BLOCK.append(f'REGISTER: {STRING_REGISTER}\n{STRING_RESPONSE_2}')
    GLOBAL_COUNTER_1 += 1
    LOCAL_COUNTER_1 += 1
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1
    LOOPS_SINCE_CHECKPOINT += 1
    IS_LAST_REGISTER = EACH_INDEX_SPECIAL == len(LIST_REGISTER_NAME) - 1
    if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or IS_LAST_REGISTER:
        FUNCTION_CHECKPOINT(EACH_INDEX_SPECIAL + 1)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 6 - CREATE - 6_REGISTER_FIELD.txt - END

3 minute(s) 38.76 second(s)

====
DONE
====


In [7]:
# 7 - CREATE - 7_REGISTER_FINAL.txt - START

"""
Merge RISC-V CSR metadata, bit fields, ISA extensions, and extracted architectural rules into a unified register specification file.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_LIST = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_RULE = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_EXTENSION = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_BIT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_READ(FILE_PATH):
    PATH_VALUE = Path(FILE_PATH)
    return PATH_VALUE.read_text(encoding='utf-8') if PATH_VALUE.exists() else ''

def FUNCTION_SPLIT_SECTION(TEXT_VALUE: str):
    MAP_OUTPUT = {}
    for EACH_PART in re.split(r'(?=^REGISTER:\s*)', TEXT_VALUE, flags=re.M):
        if not EACH_PART.strip():
            continue
        MATCH_REGISTER = re.match(r'^REGISTER:\s*(.+?)\s*[\r\n]+', EACH_PART)
        if MATCH_REGISTER:
            MAP_OUTPUT[MATCH_REGISTER.group(1).strip()] = EACH_PART.strip()
    return MAP_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
MAP_SECTION_BASE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_LIST))
MAP_SECTION_FIELD = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_BIT))
MAP_SECTION_EXTENSION = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_EXTENSION))
MAP_SECTION_RULE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_RULE))
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_REGISTER, EACH_BASE_SECTION in MAP_SECTION_BASE.items():
    LIST_LINE = EACH_BASE_SECTION.splitlines()
    if EACH_REGISTER in MAP_SECTION_FIELD:
        MATCH_FIELD = re.search(r'^\s*FIELDS:\s*(.+)$', MAP_SECTION_FIELD[EACH_REGISTER], flags=re.M)
        if MATCH_FIELD:
            STRING_FIELD = MATCH_FIELD.group(1).strip()
            for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
                if EACH_LINE.strip().startswith('FIELDS:'):
                    LIST_LINE[EACH_INDEX_LINE] = f'FIELDS: {STRING_FIELD}'
                    break
            else:
                LIST_LINE.append(f'FIELDS: {STRING_FIELD}')
    INDEX_BASE_EXTENSION = None
    LIST_EXTENSION_BASE = []
    for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
        if EACH_LINE.strip().startswith('EXTENSIONS:'):
            INDEX_BASE_EXTENSION = EACH_INDEX_LINE
            STRING_AFTER = EACH_LINE.split(':', 1)[1].strip()
            if STRING_AFTER and STRING_AFTER != '-':
                LIST_EXTENSION_BASE = [EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_AFTER.split(',') if EACH_EXTENSION.strip()]
            break
    LIST_EXTENSION_ADD = []
    if EACH_REGISTER in MAP_SECTION_EXTENSION:
        MATCH_EXTENSION = re.search(r'^\s*EXTENSIONS:\s*(.+)$', MAP_SECTION_EXTENSION[EACH_REGISTER], flags=re.M)
        if MATCH_EXTENSION:
            LIST_EXTENSION_ADD = [EACH_EXTENSION.strip() for EACH_EXTENSION in MATCH_EXTENSION.group(1).split(',') if EACH_EXTENSION.strip()]
    LIST_EXTENSION_MERGED = sorted(set(LIST_EXTENSION_BASE + LIST_EXTENSION_ADD))
    STRING_EXTENSION_LINE = 'EXTENSIONS: ' + (', '.join(LIST_EXTENSION_MERGED) if LIST_EXTENSION_MERGED else '-')
    if INDEX_BASE_EXTENSION is None:
        LIST_LINE.append(STRING_EXTENSION_LINE)
    else:
        LIST_LINE[INDEX_BASE_EXTENSION] = STRING_EXTENSION_LINE
    if EACH_REGISTER in MAP_SECTION_RULE:
        LIST_RULE_LINE = []
        for EACH_RAW in MAP_SECTION_RULE[EACH_REGISTER].splitlines():
            STRING_RAW = EACH_RAW.strip()
            if STRING_RAW.upper().startswith('RULE '):
                LIST_RULE_LINE.append(STRING_RAW.replace('“', '"').replace('”', '"'))
        if LIST_RULE_LINE:
            NUMBER_RULE_MAX = 0
            for EACH_LINE in LIST_LINE:
                MATCH_RULE = re.match(r'^\s*RULE\s+(\d+)\s*:', EACH_LINE, flags=re.I)
                if MATCH_RULE:
                    NUMBER_RULE_MAX = max(NUMBER_RULE_MAX, int(MATCH_RULE.group(1)))
            NUMBER_NEXT = NUMBER_RULE_MAX + 1
            for EACH_RULE in LIST_RULE_LINE:
                if re.match(r'^\s*RULE\s+\d+\s*:', EACH_RULE, flags=re.I):
                    LIST_LINE.append(EACH_RULE)
                else:
                    LIST_LINE.append(f'RULE {NUMBER_NEXT}: {EACH_RULE}')
                    NUMBER_NEXT += 1
    if EACH_REGISTER not in MAP_SECTION_RULE:
        continue
    LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE).strip())

Path(FILE_OUTPUT).write_text('\n\n'.join(LIST_BLOCK_FINAL), encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 7 - CREATE - 7_REGISTER_FINAL.txt - END

0.02 second(s)

====
DONE
====


In [8]:
# 8 - UPDATE - 4_REGISTER_RULE.txt - START

"""
Extend architectural rule coverage for virtual RISC-V CSRs by copying and adapting rules from their corresponding base registers while avoiding duplicate rules.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_OTHER = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_FIND_V_ALIAS(TEXT_VALUE: str):
    LIST_NAME = [EACH_MATCH.group(1).strip() for EACH_MATCH in re.finditer(r'^REGISTER:\s*([^\n]+)', TEXT_VALUE, flags=re.M)]
    SET_NAME = set(LIST_NAME)
    LIST_OUTPUT = []
    SET_SEEN = set()
    for EACH_NAME in LIST_NAME:
        if (EACH_NAME.startswith('v') and len(EACH_NAME) > 1 and EACH_NAME[1:] in SET_NAME and EACH_NAME not in SET_SEEN):
            LIST_OUTPUT.append(EACH_NAME)
            SET_SEEN.add(EACH_NAME)
    return LIST_OUTPUT

def FUNCTION_PARSE_REGISTER(TEXT_VALUE: str):
    LIST_ORDER = []
    MAP_RULE = {}
    for EACH_MATCH in REGEX_BLOCK_REGISTER.finditer(TEXT_VALUE):
        STRING_NAME = EACH_MATCH.group('name').strip()
        STRING_BODY = EACH_MATCH.group('body')
        LIST_RULE = [re.sub(r'^RULE\s+\d+:\s*', '', EACH_LINE).strip() for EACH_LINE in re.findall(r'^RULE\s+\d+:\s+.*$', STRING_BODY, flags=re.M)]
        LIST_ORDER.append(STRING_NAME)
        MAP_RULE[STRING_NAME] = LIST_RULE
    return LIST_ORDER, MAP_RULE

def FUNCTION_RENDER_REGISTER(LIST_ORDER, MAP_RULE):
    LIST_OUTPUT = []
    for EACH_NAME in LIST_ORDER:
        if EACH_NAME not in MAP_RULE:
            continue
        LIST_OUTPUT.append(f'REGISTER: {EACH_NAME}')
        LIST_OUTPUT += [f'RULE {EACH_INDEX_RULE}: {EACH_RULE}' for EACH_INDEX_RULE, EACH_RULE in enumerate(MAP_RULE[EACH_NAME], 1)]
        LIST_OUTPUT.append('')
    if LIST_OUTPUT and LIST_OUTPUT[-1] == '':
        LIST_OUTPUT.pop()
    return '\n'.join(LIST_OUTPUT)

def FUNCTION_COPY_RULE_IDEMPOTENT(STRING_BASE, STRING_TARGET, MAP_RULE):
    if STRING_BASE not in MAP_RULE or len(MAP_RULE[STRING_BASE]) < 2:
        return
    LIST_TARGET_RULE = MAP_RULE.setdefault(STRING_TARGET, [])
    REGEX_TARGET_WORD = re.compile(rf'\b{re.escape(STRING_TARGET)}\b')
    LIST_SOURCE = [EACH_RULE for EACH_RULE in MAP_RULE[STRING_BASE][1:] if not REGEX_TARGET_WORD.search(EACH_RULE)]
    if not LIST_SOURCE:
        return
    REGEX_BASE_WORD = re.compile(rf'\b{re.escape(STRING_BASE)}\b')
    LIST_CANDIDATE = [REGEX_BASE_WORD.sub(STRING_TARGET, EACH_RULE) for EACH_RULE in LIST_SOURCE]
    if LIST_CANDIDATE and LIST_CANDIDATE[0] in LIST_TARGET_RULE:
        return
    for EACH_RULE in LIST_CANDIDATE:
        if EACH_RULE not in LIST_TARGET_RULE:
            LIST_TARGET_RULE.append(EACH_RULE)

def FUNCTION_TRANSFORM(TEXT_VALUE: str, LIST_PAIR):
    LIST_ORDER, MAP_RULE = FUNCTION_PARSE_REGISTER(TEXT_VALUE)
    for EACH_BASE, EACH_TARGET in LIST_PAIR:
        if EACH_BASE in MAP_RULE and EACH_TARGET not in LIST_ORDER:
            LIST_ORDER.append(EACH_TARGET)
        FUNCTION_COPY_RULE_IDEMPOTENT(EACH_BASE, EACH_TARGET, MAP_RULE)
    return FUNCTION_RENDER_REGISTER(LIST_ORDER, MAP_RULE)
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_BLOCK_REGISTER = re.compile(r'^REGISTER:\s*(?P<name>[^\n]+)\n(?P<body>.*?)(?=^REGISTER:|\Z)', flags=re.M | re.S)
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_OTHER, encoding='utf-8') as FILE_HANDLE:
    SET_WANTED = set(re.findall(r'REGISTER:\s*(\w+)', FILE_HANDLE.read()))

with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_VALUE = FILE_HANDLE.read().strip()

LIST_PAIR = [(EACH_V_ALIAS[1:], EACH_V_ALIAS) for EACH_V_ALIAS in FUNCTION_FIND_V_ALIAS(TEXT_VALUE) if EACH_V_ALIAS in SET_WANTED]

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(FUNCTION_TRANSFORM(TEXT_VALUE, LIST_PAIR).strip())
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 8 - UPDATE - 4_REGISTER_RULE.txt - END

0.03 second(s)

====
DONE
====


In [9]:
# 9 - UPDATE - 7_REGISTER_FINAL.txt - START

"""
Update the unified RISC-V CSR specification file by merging register metadata, bit fields, ISA extensions, and updated architectural rules, including rules inherited by virtual CSRs.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_LIST = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_RULE = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_EXTENSION = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_BIT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_READ(FILE_PATH):
    PATH_VALUE = Path(FILE_PATH)
    return PATH_VALUE.read_text(encoding='utf-8') if PATH_VALUE.exists() else ''

def FUNCTION_SPLIT_SECTION(TEXT_VALUE: str):
    MAP_OUTPUT = {}
    for EACH_PART in re.split(r'(?=^REGISTER:\s*)', TEXT_VALUE, flags=re.M):
        if not EACH_PART.strip():
            continue
        MATCH_REGISTER = re.match(r'^REGISTER:\s*(.+?)\s*[\r\n]+', EACH_PART)
        if MATCH_REGISTER:
            MAP_OUTPUT[MATCH_REGISTER.group(1).strip()] = EACH_PART.strip()
    return MAP_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
MAP_SECTION_BASE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_LIST))
MAP_SECTION_FIELD = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_BIT))
MAP_SECTION_EXTENSION = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_EXTENSION))
MAP_SECTION_RULE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_RULE))
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_REGISTER, EACH_BASE_SECTION in MAP_SECTION_BASE.items():
    LIST_LINE = EACH_BASE_SECTION.splitlines()
    if EACH_REGISTER in MAP_SECTION_FIELD:
        MATCH_FIELD = re.search(r'^\s*FIELDS:\s*(.+)$', MAP_SECTION_FIELD[EACH_REGISTER], flags=re.M)
        if MATCH_FIELD:
            STRING_FIELD = MATCH_FIELD.group(1).strip()
            for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
                if EACH_LINE.strip().startswith('FIELDS:'):
                    LIST_LINE[EACH_INDEX_LINE] = f'FIELDS: {STRING_FIELD}'
                    break
            else:
                LIST_LINE.append(f'FIELDS: {STRING_FIELD}')
    INDEX_BASE_EXTENSION = None
    LIST_EXTENSION_BASE = []
    for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
        if EACH_LINE.strip().startswith('EXTENSIONS:'):
            INDEX_BASE_EXTENSION = EACH_INDEX_LINE
            STRING_AFTER = EACH_LINE.split(':', 1)[1].strip()
            if STRING_AFTER and STRING_AFTER != '-':
                LIST_EXTENSION_BASE = [EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_AFTER.split(',') if EACH_EXTENSION.strip()]
            break
    LIST_EXTENSION_ADD = []
    if EACH_REGISTER in MAP_SECTION_EXTENSION:
        MATCH_EXTENSION = re.search(r'^\s*EXTENSIONS:\s*(.+)$', MAP_SECTION_EXTENSION[EACH_REGISTER], flags=re.M)
        if MATCH_EXTENSION:
            LIST_EXTENSION_ADD = [EACH_EXTENSION.strip() for EACH_EXTENSION in MATCH_EXTENSION.group(1).split(',') if EACH_EXTENSION.strip()]
    LIST_EXTENSION_MERGED = sorted(set(LIST_EXTENSION_BASE + LIST_EXTENSION_ADD))
    STRING_EXTENSION_LINE = 'EXTENSIONS: ' + (', '.join(LIST_EXTENSION_MERGED) if LIST_EXTENSION_MERGED else '-')
    if INDEX_BASE_EXTENSION is None:
        LIST_LINE.append(STRING_EXTENSION_LINE)
    else:
        LIST_LINE[INDEX_BASE_EXTENSION] = STRING_EXTENSION_LINE
    if EACH_REGISTER in MAP_SECTION_RULE:
        LIST_RULE_LINE = []
        for EACH_RAW in MAP_SECTION_RULE[EACH_REGISTER].splitlines():
            STRING_RAW = EACH_RAW.strip()
            if STRING_RAW.upper().startswith('RULE '):
                LIST_RULE_LINE.append(STRING_RAW.replace('“', '"').replace('”', '"'))
        if LIST_RULE_LINE:
            NUMBER_RULE_MAX = 0
            for EACH_LINE in LIST_LINE:
                MATCH_RULE = re.match(r'^\s*RULE\s+(\d+)\s*:', EACH_LINE, flags=re.I)
                if MATCH_RULE:
                    NUMBER_RULE_MAX = max(NUMBER_RULE_MAX, int(MATCH_RULE.group(1)))
            NUMBER_NEXT = NUMBER_RULE_MAX + 1
            for EACH_RULE in LIST_RULE_LINE:
                if re.match(r'^\s*RULE\s+\d+\s*:', EACH_RULE, flags=re.I):
                    LIST_LINE.append(EACH_RULE)
                else:
                    LIST_LINE.append(f'RULE {NUMBER_NEXT}: {EACH_RULE}')
                    NUMBER_NEXT += 1
    if EACH_REGISTER not in MAP_SECTION_RULE:
        continue
    LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE).strip())

Path(FILE_OUTPUT).write_text('\n\n'.join(LIST_BLOCK_FINAL), encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 9 - UPDATE - 7_REGISTER_FINAL.txt - END

0.02 second(s)

====
DONE
====


In [10]:
# 10 - UPDATE 2 - 4_REGISTER_RULE.txt - START

"""
Generate additional template-based RISC-V CSR rules and assembly test cases with expected results for privilege-mode access, bit-field readback, all-ones writes, and ISA extension dependencies, then merge them with existing LLM-extracted rules.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_PARSE_BLOCK(TEXT_VALUE: str):
    LIST_BLOCK = []
    for EACH_RAW in re.split(r'\n\s*\n', TEXT_VALUE.strip()):
        MAP_VALUE = {'REGISTER': '', 'NUMBER': '', 'MODE': '', 'FIELDS': '', 'EXTENSIONS': ''}
        for EACH_LINE in EACH_RAW.splitlines():
            STRING_LINE = EACH_LINE.strip()
            if not STRING_LINE:
                continue
            for EACH_KEY in MAP_VALUE:
                if STRING_LINE.startswith(EACH_KEY + ':'):
                    MAP_VALUE[EACH_KEY] = STRING_LINE.split(':', 1)[1].strip()
                    break
        if MAP_VALUE['REGISTER'] and MAP_VALUE['MODE']:
            LIST_BLOCK.append(MAP_VALUE)
    return LIST_BLOCK

def FUNCTION_ACCESS_RULE_BY_MODE(STRING_REGISTER, STRING_MODE, STRING_HEX):
    STRING_REGISTER = STRING_REGISTER.strip()
    STRING_MODE = STRING_MODE.strip()

    if STRING_REGISTER in LIST_ILLEGAL_ALL_MODES:
        return [
            f'In M-mode, reading {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].',
            f'In S-mode, reading {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; call machine_to_supervisor; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].',
            f'In U-mode, reading {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; call machine_to_user; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].',
        ]

    if STRING_REGISTER in LIST_ILLEGAL_S_U:
        return [
            f'In M-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].',
            f'In S-mode, accessing {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; call machine_to_supervisor; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].',
            f'In U-mode, accessing {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; call machine_to_user; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].',
        ]

    if STRING_REGISTER in LIST_ILLEGAL_U_ONLY:
        return [
            f'In M-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].',
            f'In S-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; call machine_to_supervisor; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].',
            f'In U-mode, accessing {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; call machine_to_user; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].',
        ]

    STRING_RULE_BASE = f'In M-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].'
    STRING_RULE_U_ILLEGAL = f'In U-mode, accessing {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; call machine_to_user; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].'
    STRING_RULE_U_OK = f'In U-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; call machine_to_user; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].'
    STRING_SUP_MODE = STRING_MODE
    STRING_SUP_CALL = 'call machine_to_supervisor'
    STRING_RULE_SUP_OK = f'In {STRING_SUP_MODE}, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; {STRING_SUP_CALL}; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].'

    if STRING_MODE == 'M-mode':
        return [
            STRING_RULE_BASE,
            f'In S-mode, accessing {STRING_REGISTER} register should result in illegal-instruction exception (# Result: mcause = 0x2) [# Extension Must Be Active: NONE; call machine_to_supervisor; csrr a0, {STRING_HEX}; # Result: mcause = 0x2].',
            STRING_RULE_U_ILLEGAL
        ]

    if STRING_MODE == 'U-mode':
        return [
            STRING_RULE_BASE,
            f'In S-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; call machine_to_supervisor; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].',
            STRING_RULE_U_OK
        ]

    if STRING_MODE in ('HS-mode', 'VS-mode', 'S-mode'):
        if STRING_MODE == 'HS-mode':
            STRING_RULE_SUP_OK = f'In HS-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; call machine_to_supervisor; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].'
        elif STRING_MODE == 'VS-mode':
            STRING_RULE_SUP_OK = f'In VS-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; call machine_to_virtual; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].'
        else:
            STRING_RULE_SUP_OK = f'In S-mode, accessing {STRING_REGISTER} register should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: NONE; call machine_to_supervisor; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].'
        return [STRING_RULE_BASE, STRING_RULE_SUP_OK, STRING_RULE_U_ILLEGAL]

    return [STRING_RULE_BASE, STRING_RULE_SUP_OK, STRING_RULE_U_ILLEGAL]

def FUNCTION_PARSE_FIELD(STRING_FIELD):
    LIST_OUTPUT = []
    if not STRING_FIELD:
        return LIST_OUTPUT
    for EACH_PART in [EACH_ITEM.strip() for EACH_ITEM in STRING_FIELD.split(',') if EACH_ITEM.strip()]:
        MATCH_FIELD = re.match(r'(\d+)\.\.(\d+)\s*=\s*([A-Za-z0-9_]+)', EACH_PART)
        if MATCH_FIELD:
            NUMBER_HIGH = int(MATCH_FIELD.group(1))
            NUMBER_LOW = int(MATCH_FIELD.group(2))
            STRING_NAME = MATCH_FIELD.group(3)
            LIST_OUTPUT.append({'name': STRING_NAME, 'bit': min(NUMBER_HIGH, NUMBER_LOW)})
    return LIST_OUTPUT

def FUNCTION_FIELD_RULE(STRING_REGISTER, STRING_FIELD, STRING_HEX):
    LIST_RULE = []
    for EACH_FIELD in FUNCTION_PARSE_FIELD(STRING_FIELD):
        NUMBER_BIT = EACH_FIELD['bit']
        STRING_FIELD_NAME = EACH_FIELD['name']
        STRING_MASK_HEX = hex(1 << NUMBER_BIT)
        TUPLE_KEY = (STRING_REGISTER, STRING_FIELD_NAME, NUMBER_BIT)
        if TUPLE_KEY in SET_SKIP_FIELD_RULE:
            continue
        if TUPLE_KEY in SET_FORCE_ZERO_READBACK_FIELD:
            STRING_EXPECT = '0x0'
            STRING_MUST = 'should not'
        else:
            STRING_EXPECT = STRING_MASK_HEX
            STRING_MUST = 'should'
        LIST_RULE.append(
            f'In M-mode, setting {STRING_REGISTER}.{STRING_FIELD_NAME} (bit {NUMBER_BIT}) '
            f'{STRING_MUST} be reflected on the readback '
            f'(# Result: a2 = {STRING_EXPECT}) '
            f'[# Extension Must Be Active: NONE; li a0, {STRING_MASK_HEX}; csrs {STRING_HEX}, a0; '
            f'csrr a1, {STRING_HEX}; and a2, a0, a1; # Result: a2 = {STRING_EXPECT}].'
        )
    return LIST_RULE

def FUNCTION_WRITE_MINUS_ONE_RULE(STRING_REGISTER, STRING_HEX):
    return f'In M-mode, writing 0xffffffffffffffff to the {STRING_REGISTER} register should not set it to 0xffffffff or 0xffffffffffffffff (# Result: a1 != 0xffffffff & a1 != 0xffffffffffffffff) [# Extension Must Be Active: NONE; li a0, 0xffffffffffffffff; csrw {STRING_HEX}, a0; csrr a1, {STRING_HEX}; # Result: a1 != 0xffffffff & a1 != 0xffffffffffffffff].'

def FUNCTION_EXTENSION_RULE(STRING_REGISTER, STRING_EXTENSION, STRING_HEX):
    if not STRING_EXTENSION.strip():
        return []
    LIST_EXTENSION = [EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_EXTENSION.split(',') if EACH_EXTENSION.strip() and '-' not in EACH_EXTENSION]
    return [f'In M-mode, accessing {STRING_REGISTER} register while {EACH_EXTENSION} extension is active should result in no exception (# Result: mcause = 0x0) [# Extension Must Be Active: {EACH_EXTENSION}; csrr a0, {STRING_HEX}; # Result: mcause = 0x0].' for EACH_EXTENSION in LIST_EXTENSION]

def FUNCTION_GENERATE_RULE(TEXT_VALUE: str):
    LIST_OUTPUT = []
    for EACH_BLOCK in FUNCTION_PARSE_BLOCK(TEXT_VALUE):
        STRING_REGISTER = EACH_BLOCK['REGISTER']
        STRING_HEX = EACH_BLOCK['NUMBER']
        STRING_MODE = EACH_BLOCK['MODE']
        STRING_FIELD = EACH_BLOCK.get('FIELDS', '')
        STRING_EXTENSION = EACH_BLOCK.get('EXTENSIONS', '')
        LIST_OUTPUT.append(f'REGISTER: {STRING_REGISTER}')
        LIST_RULE = []
        if STRING_REGISTER not in SET_SKIP_ACCESS_RULE_BY_MODE:
            LIST_RULE += FUNCTION_ACCESS_RULE_BY_MODE(STRING_REGISTER, STRING_MODE, STRING_HEX)
        if STRING_REGISTER not in SET_EXCLUDED_FOR_FIELD_RULE and STRING_FIELD:
            LIST_RULE += FUNCTION_FIELD_RULE(STRING_REGISTER, STRING_FIELD, STRING_HEX)
        if STRING_REGISTER not in SET_EXCLUDED_FOR_WRITE_MINUS_ONE_RULE and STRING_FIELD:
            LIST_RULE.append(FUNCTION_WRITE_MINUS_ONE_RULE(STRING_REGISTER, STRING_HEX))
        if STRING_REGISTER != 'seed' and STRING_EXTENSION:
            LIST_RULE += FUNCTION_EXTENSION_RULE(STRING_REGISTER, STRING_EXTENSION, STRING_HEX)
        LIST_OUTPUT += [f'RULE {EACH_INDEX}: {EACH_RULE}' for EACH_INDEX, EACH_RULE in enumerate(LIST_RULE, 1)]
        LIST_OUTPUT.append('')
    if LIST_OUTPUT and LIST_OUTPUT[-1] == '':
        LIST_OUTPUT.pop()
    return '\n'.join(LIST_OUTPUT)

def FUNCTION_MERGE_RULE(TEXT_OTHER: str, TEXT_NEW_RULE: str):
    TEXT_MERGED = TEXT_OTHER
    def FUNCTION_FIND_BLOCK(TEXT_VALUE, STRING_NAME):
        return re.search(rf'^REGISTER:\s*{re.escape(STRING_NAME)}\n(?P<body>.*?)(?=^REGISTER:|\Z)', TEXT_VALUE, flags=re.M | re.S)
    for EACH_MATCH in REGEX_BLOCK_REGISTER.finditer(TEXT_NEW_RULE):
        STRING_REGISTER = EACH_MATCH.group('name').strip()
        LIST_NEW_RULE_LINE = re.findall(r'^RULE\s+\d+:\s+.*$', EACH_MATCH.group('body'), flags=re.M)
        MATCH_EXIST = FUNCTION_FIND_BLOCK(TEXT_MERGED, STRING_REGISTER)
        if MATCH_EXIST:
            STRING_BLOCK = MATCH_EXIST.group(0)
            LIST_NUMBER = re.findall(r'^RULE\s+(\d+):', STRING_BLOCK, flags=re.M)
            NUMBER_LAST = int(LIST_NUMBER[-1]) if LIST_NUMBER else 0
            LIST_APPEND = []
            for EACH_INDEX, EACH_RULE in enumerate(LIST_NEW_RULE_LINE, 1):
                RULE_TEXT = re.sub(r'^RULE\s+\d+:\s*', '', EACH_RULE).strip()
                LIST_APPEND.append(f'RULE {NUMBER_LAST + EACH_INDEX}: {RULE_TEXT}')
            NUMBER_START, NUMBER_END = MATCH_EXIST.span()
            TEXT_MERGED = TEXT_MERGED[:NUMBER_START] + (STRING_BLOCK.rstrip() + ('\n' + '\n'.join(LIST_APPEND) if LIST_APPEND else '') + '\n\n') + TEXT_MERGED[NUMBER_END:]
        else:
            STRING_BLOCK_TEXT = EACH_MATCH.group(0).strip()
            TEXT_MERGED = (TEXT_MERGED.rstrip() + '\n\n' + STRING_BLOCK_TEXT + '\n\n') if TEXT_MERGED.strip() else (STRING_BLOCK_TEXT + '\n\n')
    TEXT_MERGED = re.sub(r'(?m)^(REGISTER:)', r'\n\1', TEXT_MERGED).lstrip()
    TEXT_MERGED = re.sub(r'\n{3,}', '\n\n', TEXT_MERGED).rstrip() + '\n'
    return TEXT_MERGED
# FUNCTION - FINISH

# VARIABLE - BEGIN
TURN_ON_PYTHON_RULES = True
REGEX_BLOCK_REGISTER = re.compile(r'^REGISTER:\s*(?P<name>[^\n]+)\n(?P<body>.*?)(?=^REGISTER:|\Z)', flags=re.M | re.S)
SET_EXCLUDED_FOR_FIELD_RULE = {'vtype', 'vlenb', 'vl', 'mie', 'seed', 'mscratch', 'sscratch', 'vsscratch', 'mnscratch', 'dscratch0', 'dscratch1', 'mscratchcsw', 'mscratchcswl', 'sscratchcsw', 'sscratchcswl', 'uscratchcsw', 'uscratchcswl', 'mvendorid', 'marchid', 'mimpid', 'mhartid', 'htimedeltah', 'stimecmp', 'stimecmph', 'vstimecmp', 'timeh', 'vstimecmph', 'htimedelta', 'time', 'cycle', 'instret', 'mcycle', 'minstret', 'cycleh', 'mcyclecfg', 'mcyclecfgh', 'mcycleh', 'minstretcfg', 'minstretcfgh', 'minstreth', 'instreth', 'hpmcounter3', 'hpmcounter4', 'hpmcounter5', 'hpmcounter6', 'hpmcounter7', 'hpmcounter8', 'hpmcounter9', 'hpmcounter10', 'hpmcounter11', 'hpmcounter12', 'hpmcounter13', 'hpmcounter14', 'hpmcounter15', 'hpmcounter16', 'hpmcounter17', 'hpmcounter18', 'hpmcounter19', 'hpmcounter20', 'hpmcounter21', 'hpmcounter22', 'hpmcounter23', 'hpmcounter24', 'hpmcounter25', 'hpmcounter26', 'hpmcounter27', 'hpmcounter28', 'hpmcounter29', 'hpmcounter30', 'hpmcounter31', 'hpmcounter3h', 'hpmcounter4h', 'hpmcounter5h', 'hpmcounter6h', 'hpmcounter7h', 'hpmcounter8h', 'hpmcounter9h', 'hpmcounter10h', 'hpmcounter11h', 'hpmcounter12h', 'hpmcounter13h', 'hpmcounter14h', 'hpmcounter15h', 'hpmcounter16h', 'hpmcounter17h', 'hpmcounter18h', 'hpmcounter19h', 'hpmcounter20h', 'hpmcounter21h', 'hpmcounter22h', 'hpmcounter23h', 'hpmcounter24h', 'hpmcounter25h', 'hpmcounter26h', 'hpmcounter27h', 'hpmcounter28h', 'hpmcounter29h', 'hpmcounter30h', 'hpmcounter31h', 'mhpmcounter3', 'mhpmcounter4', 'mhpmcounter5', 'mhpmcounter6', 'mhpmcounter7', 'mhpmcounter8', 'mhpmcounter9', 'mhpmcounter10', 'mhpmcounter11', 'mhpmcounter12', 'mhpmcounter13', 'mhpmcounter14', 'mhpmcounter15', 'mhpmcounter16', 'mhpmcounter17', 'mhpmcounter18', 'mhpmcounter19', 'mhpmcounter20', 'mhpmcounter21', 'mhpmcounter22', 'mhpmcounter23', 'mhpmcounter24', 'mhpmcounter25', 'mhpmcounter26', 'mhpmcounter27', 'mhpmcounter28', 'mhpmcounter29', 'mhpmcounter30', 'mhpmcounter31', 'mhpmcounter3h', 'mhpmcounter4h', 'mhpmcounter5h', 'mhpmcounter6h', 'mhpmcounter7h', 'mhpmcounter8h', 'mhpmcounter9h', 'mhpmcounter10h', 'mhpmcounter11h', 'mhpmcounter12h', 'mhpmcounter13h', 'mhpmcounter14h', 'mhpmcounter15h', 'mhpmcounter16h', 'mhpmcounter17h', 'mhpmcounter18h', 'mhpmcounter19h', 'mhpmcounter20h', 'mhpmcounter21h', 'mhpmcounter22h', 'mhpmcounter23h', 'mhpmcounter24h', 'mhpmcounter25h', 'mhpmcounter26h', 'mhpmcounter27h', 'mhpmcounter28h', 'mhpmcounter29h', 'mhpmcounter30h', 'mhpmcounter31h'}

SET_SKIP_ACCESS_RULE_BY_MODE = {
    'scountinhibit',
}

SET_SKIP_FIELD_RULE = {
    ('medeleg', 'EM', 11),
    ('tdata1', 'DATA', 0),
    ('tdata3', 'DATA', 0),
    ('hgatp', 'PPN', 0),
    ('hstatus', 'VGEIN', 12),
    ('misa', 'B', 1),
    ('mstatus', 'UXL', 32),
    ('sstatus', 'UXL', 32),
    ('hcounteren', 'HPM19', 19),
    ('hcounteren', 'HPM20', 20),
    ('hcounteren', 'HPM21', 21),
    ('hcounteren', 'HPM22', 22),
    ('hcounteren', 'HPM23', 23),
    ('hcounteren', 'HPM24', 24),
    ('hcounteren', 'HPM25', 25),
    ('hcounteren', 'HPM26', 26),
    ('hcounteren', 'HPM27', 27),
    ('hcounteren', 'HPM28', 28),
    ('hcounteren', 'HPM29', 29),
    ('hcounteren', 'HPM30', 30),
    ('hcounteren', 'HPM31', 31),
    ('henvcfg', 'FIOM', 0),
    ('htinst', 'VALUE', 0),

    ('mcounteren', 'HPM31', 31),
    ('mcounteren', 'HPM30', 30),
    ('mcounteren', 'HPM29', 29),
    ('mcounteren', 'HPM28', 28),
    ('mcounteren', 'HPM27', 27),
    ('mcounteren', 'HPM26', 26),
    ('mcounteren', 'HPM25', 25),
    ('mcounteren', 'HPM24', 24),
    ('mcounteren', 'HPM23', 23),
    ('mcounteren', 'HPM22', 22),
    ('mcounteren', 'HPM21', 21),
    ('mcounteren', 'HPM20', 20),
    ('mcounteren', 'HPM19', 19),
    
    ('mcountinhibit', 'HPM31', 31),
    ('mcountinhibit', 'HPM30', 30),
    ('mcountinhibit', 'HPM29', 29),
    ('mcountinhibit', 'HPM28', 28),
    ('mcountinhibit', 'HPM27', 27),
    ('mcountinhibit', 'HPM26', 26),
    ('mcountinhibit', 'HPM25', 25),
    ('mcountinhibit', 'HPM24', 24),
    ('mcountinhibit', 'HPM23', 23),
    ('mcountinhibit', 'HPM22', 22),
    ('mcountinhibit', 'HPM21', 21),
    ('mcountinhibit', 'HPM20', 20),
    ('mcountinhibit', 'HPM19', 19),
    
    ('menvcfg', 'FIOM', 0),
    
    ('mhpmevent3', 'EVENT', 0),
    ('mhpmevent4', 'EVENT', 0),
    ('mhpmevent5', 'EVENT', 0),
    ('mhpmevent6', 'EVENT', 0),
    ('mhpmevent7', 'EVENT', 0),
    ('mhpmevent8', 'EVENT', 0),
    ('mhpmevent9', 'EVENT', 0),
    ('mhpmevent10', 'EVENT', 0),
    ('mhpmevent11', 'EVENT', 0),
    ('mhpmevent12', 'EVENT', 0),
    ('mhpmevent13', 'EVENT', 0),
    ('mhpmevent14', 'EVENT', 0),
    ('mhpmevent15', 'EVENT', 0),
    ('mhpmevent16', 'EVENT', 0),
    ('mhpmevent17', 'EVENT', 0),
    ('mhpmevent18', 'EVENT', 0),
    ('mhpmevent19', 'EVENT', 0),
    ('mhpmevent20', 'EVENT', 0),
    ('mhpmevent21', 'EVENT', 0),
    ('mhpmevent22', 'EVENT', 0),
    ('mhpmevent23', 'EVENT', 0),
    ('mhpmevent24', 'EVENT', 0),
    ('mhpmevent25', 'EVENT', 0),
    ('mhpmevent26', 'EVENT', 0),
    ('mhpmevent27', 'EVENT', 0),
    ('mhpmevent28', 'EVENT', 0),
    ('mhpmevent29', 'EVENT', 0),
    ('mhpmevent30', 'EVENT', 0),
    ('mhpmevent31', 'EVENT', 0),
    
    ('mip', 'VSTIP', 6),
    ('mip', 'VSEIP', 10),
    
    ('mseccfg', 'WPRI', 34),
    ('mseccfg', 'PMM', 32),
    ('mseccfg', 'WPRI', 11),
    ('mseccfg', 'MLPE', 10),
    ('mseccfg', 'WPRI', 3),
    ('mseccfg', 'MML', 0),
    
    ('mstateen0', 'AIA', 59),
    ('mstateen0', 'IMSIC', 58),
    ('mstateen0', 'P1P13', 56),
    ('mstateen0', 'C', 0),
    
    ('mstateen0h', 'SE0', 31),
    ('mstateen0h', 'ENVCFG', 30),
    ('mstateen0h', 'P1P13', 24),
    
    ('mstateen1h', 'SE0', 31),
    ('mstateen2h', 'SE0', 31),
    ('mstateen3h', 'SE0', 31),
    
    ('scounteren', 'HPM31', 31),
    ('scounteren', 'HPM30', 30),
    ('scounteren', 'HPM29', 29),
    ('scounteren', 'HPM28', 28),
    ('scounteren', 'HPM27', 27),
    ('scounteren', 'HPM26', 26),
    ('scounteren', 'HPM25', 25),
    ('scounteren', 'HPM24', 24),
    ('scounteren', 'HPM23', 23),
    ('scounteren', 'HPM22', 22),
    ('scounteren', 'HPM21', 21),
    ('scounteren', 'HPM20', 20),
    ('scounteren', 'HPM19', 19),
    
    ('senvcfg', 'FIOM', 0),
    
    ('henvcfg', 'ADUE', 61),
    ('menvcfg', 'ADUE', 61),
}

SET_FORCE_ZERO_READBACK_FIELD = {
    ('hgatp', 'MODE', 60),
    ('satp', 'MODE', 60),

    *{('hcounteren', f'HPM{i}', i) for i in range(19, 32)},
    ('hedeleg', 'SGPF', 23),
    ('hedeleg', 'VI', 22),
    ('hedeleg', 'LGPF', 21),
    ('hedeleg', 'IGPF', 20),
    ('hedeleg', 'EM', 11),
    ('hedeleg', 'EVS', 10),
    ('hedeleg', 'ES', 9),
    ('henvcfg', 'STCE', 63),
    ('henvcfg', 'PBMTE', 62),

    ('hstateen0', 'SE0', 63),
    ('hstateen0', 'ENVCFG', 62),
    ('hstateen0', 'CSRIND', 60),
    ('hstateen0', 'AIA', 59),
    ('hstateen0', 'IMSIC', 58),
    ('hstateen0', 'CONTEXT', 57),
    ('hstateen0', 'CTR', 54),
    ('hstateen0', 'JVT', 2),
    ('hstateen0', 'FCSR', 1),
    ('hstateen0', 'C', 0),

    ('hstateen0h', 'SE0', 31),
    ('hstateen0h', 'ENVCFG', 30),
    ('hstateen0h', 'CSRIND', 28),
    ('hstateen0h', 'AIA', 27),
    ('hstateen0h', 'IMSIC', 26),
    ('hstateen0h', 'CONTEXT', 25),
    ('hstateen0h', 'CTR', 22),

    ('hstateen1', 'SE0', 63),
    ('hstateen1h', 'SE0', 31),
    ('hstateen2', 'SE0', 63),
    ('hstateen2h', 'SE0', 31),
    ('hstateen3', 'SE0', 63),
    ('hstateen3h', 'SE0', 31),

    ('hstatus', 'VSXL', 32),
    ('hstatus', 'VSBE', 5),
    ('htinst', 'VALUE', 0),
    ('mconfigptr', 'ADDRESS', 0),

    *{('mcounteren', f'HPM{i}', i) for i in range(19, 32)},
    *{('mcountinhibit', f'HPM{i}', i) for i in range(19, 32)},

    ('menvcfg', 'PBMTE', 62),
    ('mepc', 'PC', 0),
    ('mideleg', 'MSI', 3),
    ('mideleg', 'MTI', 7),
    ('mideleg', 'MEI', 11),
    ('mip', 'MSIP', 3),
    ('mip', 'MEIP', 11),
    ('mip', 'SGEIP', 12),

    ('misa', 'MXL', 62),
    ('misa', 'B', 1),
    ('misa', 'G', 6),
    ('misa', 'Q', 16),

    ('mseccfg', 'MML', 0),

    ('mstateen0', 'CONTEXT', 57),
    ('mstateen0', 'SRMCFG', 55),
    ('mstateen0', 'CTR', 54),
    ('mstateen0', 'JVT', 2),
    ('mstateen0', 'FCSR', 1),
    ('mstateen0', 'C', 0),

    ('mstateen0h', 'CSRIND', 28),
    ('mstateen0h', 'AIA', 27),
    ('mstateen0h', 'IMSIC', 26),
    ('mstateen0h', 'CONTEXT', 25),
    ('mstateen0h', 'SRMCFG', 23),
    ('mstateen0h', 'CTR', 22),

    ('mstatus', 'MDT', 42), # Smdbltrp
    ('mstatus', 'MBE', 37),
    ('mstatus', 'SBE', 36),
    ('mstatus', 'SXL', 34),
    ('mstatus', 'XS', 15),
    ('mstatus', 'UBE', 6),

    *{(f'pmpaddr{i}', 'ADDR', 0) for i in range(16, 64)},

    ('pmpcfg4', 'pmp16cfg', 0),
    ('pmpcfg4', 'pmp17cfg', 8),
    ('pmpcfg4', 'pmp18cfg', 16),
    ('pmpcfg4', 'pmp19cfg', 24),
    ('pmpcfg4', 'pmp20cfg', 32),
    ('pmpcfg4', 'pmp21cfg', 40),
    ('pmpcfg4', 'pmp22cfg', 48),
    ('pmpcfg4', 'pmp23cfg', 56),

    ('pmpcfg6', 'pmp24cfg', 0),
    ('pmpcfg6', 'pmp25cfg', 8),
    ('pmpcfg6', 'pmp26cfg', 16),
    ('pmpcfg6', 'pmp27cfg', 24),
    ('pmpcfg6', 'pmp28cfg', 32),
    ('pmpcfg6', 'pmp29cfg', 40),
    ('pmpcfg6', 'pmp30cfg', 48),
    ('pmpcfg6', 'pmp31cfg', 56),

    ('pmpcfg8', 'pmp32cfg', 0),
    ('pmpcfg8', 'pmp33cfg', 8),
    ('pmpcfg8', 'pmp34cfg', 16),
    ('pmpcfg8', 'pmp35cfg', 24),
    ('pmpcfg8', 'pmp36cfg', 32),
    ('pmpcfg8', 'pmp37cfg', 40),
    ('pmpcfg8', 'pmp38cfg', 48),
    ('pmpcfg8', 'pmp39cfg', 56),

    ('pmpcfg10', 'pmp40cfg', 0),
    ('pmpcfg10', 'pmp41cfg', 8),
    ('pmpcfg10', 'pmp42cfg', 16),
    ('pmpcfg10', 'pmp43cfg', 24),
    ('pmpcfg10', 'pmp44cfg', 32),
    ('pmpcfg10', 'pmp45cfg', 40),
    ('pmpcfg10', 'pmp46cfg', 48),
    ('pmpcfg10', 'pmp47cfg', 56),

    ('pmpcfg12', 'pmp48cfg', 0),
    ('pmpcfg12', 'pmp49cfg', 8),
    ('pmpcfg12', 'pmp50cfg', 16),
    ('pmpcfg12', 'pmp51cfg', 24),
    ('pmpcfg12', 'pmp52cfg', 32),
    ('pmpcfg12', 'pmp53cfg', 40),
    ('pmpcfg12', 'pmp54cfg', 48),
    ('pmpcfg12', 'pmp55cfg', 56),

    ('pmpcfg14', 'pmp56cfg', 0),
    ('pmpcfg14', 'pmp57cfg', 8),
    ('pmpcfg14', 'pmp58cfg', 16),
    ('pmpcfg14', 'pmp59cfg', 24),
    ('pmpcfg14', 'pmp60cfg', 32),
    ('pmpcfg14', 'pmp61cfg', 40),
    ('pmpcfg14', 'pmp62cfg', 48),
    ('pmpcfg14', 'pmp63cfg', 56),

    *{('scounteren', f'HPM{i}', i) for i in range(19, 32)},
    *{('scountovf', f'OF{i}', i) for i in range(3, 32)},

    ('sepc', 'PC', 0),
    ('sip', 'STIP', 5),
    ('sip', 'SEIP', 9),

    ('sstateen0', 'JVT', 2),
    ('sstateen0', 'FCSR', 1),
    ('sstateen0', 'C', 0),
    ('sstateen1', 'DATA', 0),
    ('sstateen2', 'DATA', 0),
    ('sstateen3', 'DATA', 0),

    ('sstatus', 'XS', 15),
    ('sstatus', 'UBE', 6),

    ('tdata1', 'DMODE', 59),
    ('tdata3', 'DATA', 0),

    ('vl', 'VALUE', 0),
    ('vlenb', 'VALUE', 0),
    ('vtype', 'VILL', 63),
    ('vtype', 'VMA', 7),
    ('vtype', 'VTA', 6),
    ('vtype', 'VSEW', 3),
    ('vtype', 'VLMUL', 0),
}

LIST_EXCLUDED_FOR_WRITE_MINUS_ONE_RULE = [
    'htval',
    'hviprio1',
    'hviprio2',
    'mcause',
    *[f'mhpmevent{i}' for i in range(3, 32)],
    *[f'mhpmcounter{i}' for i in range(3, 32)],
    'mtinst',
    'mtval',
    'mtval2',
    *[f'pmpaddr{i}' for i in range(64)],
    *[f'pmpcfg{i}' for i in range(0, 15, 2)],
    'scause',
    'ssp',
    'stval',
    'tdata2',
    'tdata1',
    'vxrm',
    'vstval',
    'seed',
    'htimedelta',
    'mscratch',
    'sscratch',
    'vsscratch',
    'stimecmp',
    'vstimecmp',
    'vtype',
    'vlenb',
    'vl',
    'mie',
    'hcounteren',
    'htinst',
    'mcounteren',
    'scounteren',
    'mcycle',
    'minstret',
    'miselect',
    'mseccfg',
    'siselect',
]

SET_EXCLUDED_FOR_WRITE_MINUS_ONE_RULE = set(LIST_EXCLUDED_FOR_WRITE_MINUS_ONE_RULE)

LIST_ILLEGAL_ALL_MODES = [
    'seed',
]

LIST_ILLEGAL_S_U = [
    'cycle',
    *[f'hpmcounter{i}' for i in range(3, 19)],
    'instret',
    'time',
    'ssp',
    'tinfo',
    'vstimecmp',
    'vstopi',
    'henvcfg',
    *[f'hstateen{i}' for i in range(4)],
    *[f'hstateen{i}h' for i in range(4)],
    'hvictl',
    'hvien',
    'hviprio1',
    'hviprio2',
    'senvcfg',
    'siselect',
    *[f'sstateen{i}' for i in range(4)],
    'stimecmp',
    'stopi',
]

LIST_ILLEGAL_U_ONLY = [
    'vsie',
    'vsip',
    'vsscratch',
    'htval',
]
# VARIABLE - FINISH

# CODE - BEGIN
if TURN_ON_PYTHON_RULES:
    with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_OTHER = FILE_HANDLE.read().strip()
    
    with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_VALUE = FILE_HANDLE.read().strip()
    
    TEXT_CLEANED = re.sub(r'^(WIDTH:|RULE\s*\d+:).*$', '', TEXT_VALUE, flags=re.M)
    TEXT_CLEANED = re.sub(r'\n\s*\n', '\n', TEXT_CLEANED)
    TEXT_CLEANED = re.sub(r'(REGISTER: [^\n]+)', r'\n\1', TEXT_CLEANED).strip()
    TEXT_NEW_RULE = FUNCTION_GENERATE_RULE(TEXT_CLEANED)
    LIST_NEW_RULE_LINE = TEXT_NEW_RULE.splitlines()
    STRING_FIRST = re.sub(r'^RULE\s+\d+:\s*', '', LIST_NEW_RULE_LINE[1])
    
    if STRING_FIRST not in TEXT_OTHER:
        with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.write(FUNCTION_MERGE_RULE(TEXT_OTHER, TEXT_NEW_RULE).strip())
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 10 - UPDATE 2 - 4_REGISTER_RULE.txt - END

0.60 second(s)

====
DONE
====


In [ ]:
# 11 - UPDATE 2 - 7_REGISTER_FINAL.txt - START

"""
Merge RISC-V CSR metadata with updated architectural rules, filter rules based on the target RV64 configuration and test harness capabilities, and refine them using LLMs for automatic assembly test case generation while preserving applicable template-based rules and their assembly test cases.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
from ollama import chat, Client
# IMPORT - FINISH

# PATH - BEGIN
FILE_LIST = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_RULE = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_EXTENSION = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_BIT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_READ(FILE_PATH):
    PATH_VALUE = Path(FILE_PATH)
    return PATH_VALUE.read_text(encoding='utf-8') if PATH_VALUE.exists() else ''

def FUNCTION_SPLIT_SECTION(TEXT_VALUE: str):
    MAP_OUTPUT = {}
    for EACH_PART in re.split(r'(?=^REGISTER:\s*)', TEXT_VALUE, flags=re.M):
        if not EACH_PART.strip():
            continue
        MATCH_REGISTER = re.match(r'^REGISTER:\s*(.+?)\s*[\r\n]+', EACH_PART)
        if MATCH_REGISTER:
            MAP_OUTPUT[MATCH_REGISTER.group(1).strip()] = EACH_PART.strip()
    return MAP_OUTPUT

def FUNCTION_CALL_LLM(STRING_PROMPT):
    if IS_LOCAL:
        RESPONSE = chat(
            model=MODEL,
            messages=[{'role': 'user', 'content': STRING_PROMPT}],
            options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT},
            keep_alive='30m'
        ).message.content
    else:
        RESPONSE = CLIENT.chat(
            model=MODEL,
            messages=[{'role': 'user', 'content': STRING_PROMPT}],
            options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT},
            keep_alive='30m'
        ).message.content
    return RESPONSE.strip()

def FUNCTION_CHECK_RULE_TESTABLE(STRING_RULE):
    STRING_PROMPT = f'''You are given one RISC-V architectural rule.

Determine whether the rule is already clear, directly testable, and actionable for automatic assembly test generation.

Output only:
YES
or
NO

Output YES only if:
- the rule contains a clear architectural condition or setup
- the rule contains a clear observable architectural result
- the observable result can be tested using assembly
- the rule is specific enough to generate a concrete test case
- the rule describes behavior observable through:
  - registers
  - CSRs
  - trap behavior
  - mcause/scause
  - privilege behavior
  - instruction result
  - readback behavior
  - architectural exceptions

Output NO if:
- the rule is vague or descriptive
- the rule only describes metadata
- the rule states a constraint without an observable consequence
- the rule lacks a concrete expected result
- the rule is mainly about:
  - implementation details
  - microarchitectural behavior
  - performance
  - timing
  - cache/TLB internals
  - synchronization internals
  - implicit hardware behavior
- the rule is too abstract to generate a concrete assembly test

The rule:
{STRING_RULE}'''

    STRING_RESPONSE = FUNCTION_CALL_LLM(STRING_PROMPT)
    return STRING_RESPONSE.strip().upper().startswith('YES')

def FUNCTION_REWRITE_RULE_TESTABLE(STRING_RULE):
    STRING_PROMPT = f'''You are given one old RISC-V architectural rule.

Rewrite it into one clearer, more actionable, directly testable rule for assembly test generation.

Output only the rewritten rule. No explanation. No bullet. No numbering.

Requirements:
- Keep the original meaning.
- Make the rule observable through architectural state when possible.
- Prefer violation-based cause-effect wording:
  "If <required condition is violated>, then <observable architectural result>."
- Do not output rules in the form "<thing> must be <constraint>" unless the observable consequence is also stated.
- For constraint rules, explicitly state what happens when the constraint is violated.
- If violating a constraint causes an exception, explicitly mention the exception.
- Prefer concrete observable outcomes such as "raises an illegal-instruction exception", "sets mcause to ...", "the CSR readback value is ...", or "the destination register contains ...".
- Make implicit constraints explicit.
- If the old rule is already clear and directly testable, keep it mostly unchanged.
- If the old rule only states metadata or a constraint, rewrite it into the architectural consequence of violating or satisfying that constraint.
- Do not invent unrelated behavior.
- Do not make the rule more specific than the old rule unless needed to make it testable.

Old rule:
{STRING_RULE}'''

    STRING_RESPONSE = FUNCTION_CALL_LLM(STRING_PROMPT)
    STRING_RESPONSE = STRING_RESPONSE.strip()

    STRING_RESPONSE = re.sub(r'^```(?:\w+)?\s*', '', STRING_RESPONSE)
    STRING_RESPONSE = re.sub(r'\s*```$', '', STRING_RESPONSE)
    STRING_RESPONSE = STRING_RESPONSE.strip()

    STRING_RESPONSE = re.sub(r'(?i)^RULE\s+\d+\s*:\s*', '', STRING_RESPONSE).strip()

    return STRING_RESPONSE

def FUNCTION_REWRITE_FINAL_REGISTER_RULES(LIST_BLOCK_FINAL):
    LIST_BLOCK_REWRITTEN = []

    for EACH_BLOCK in LIST_BLOCK_FINAL:
        LIST_LINE_REWRITTEN = []

        for EACH_LINE in EACH_BLOCK.splitlines():
            MATCH_RULE = re.match(r'^(RULE\s+\d+:\s*)(.+?)\s*$', EACH_LINE)

            if not MATCH_RULE:
                LIST_LINE_REWRITTEN.append(EACH_LINE)
                continue

            STRING_RULE_PREFIX = MATCH_RULE.group(1)
            STRING_OLD_RULE = MATCH_RULE.group(2).strip()

            if '[' in STRING_OLD_RULE and ']' in STRING_OLD_RULE:
                LIST_LINE_REWRITTEN.append(EACH_LINE)
                continue

            IS_TESTABLE = FUNCTION_CHECK_RULE_TESTABLE(STRING_OLD_RULE)

            if IS_TESTABLE:
                LIST_LINE_REWRITTEN.append(EACH_LINE)
                continue

            STRING_NEW_RULE = FUNCTION_REWRITE_RULE_TESTABLE(STRING_OLD_RULE)

            if not STRING_NEW_RULE:
                STRING_NEW_RULE = STRING_OLD_RULE

            LIST_LINE_REWRITTEN.append(f'{STRING_RULE_PREFIX}{STRING_NEW_RULE}')

        LIST_BLOCK_REWRITTEN.append('\n'.join(LIST_LINE_REWRITTEN).strip())

    return LIST_BLOCK_REWRITTEN
# FUNCTION - FINISH

# VARIABLE - BEGIN
MAP_SECTION_BASE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_LIST))
MAP_SECTION_FIELD = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_BIT))
MAP_SECTION_EXTENSION = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_EXTENSION))
MAP_SECTION_RULE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_RULE))
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_REGISTER, EACH_BASE_SECTION in MAP_SECTION_BASE.items():
    LIST_LINE = EACH_BASE_SECTION.splitlines()

    if EACH_REGISTER in MAP_SECTION_FIELD:
        MATCH_FIELD = re.search(r'^\s*FIELDS:\s*(.+)$', MAP_SECTION_FIELD[EACH_REGISTER], flags=re.M)
        if MATCH_FIELD:
            STRING_FIELD = MATCH_FIELD.group(1).strip()
            for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
                if EACH_LINE.strip().startswith('FIELDS:'):
                    LIST_LINE[EACH_INDEX_LINE] = f'FIELDS: {STRING_FIELD}'
                    break
            else:
                LIST_LINE.append(f'FIELDS: {STRING_FIELD}')

    INDEX_BASE_EXTENSION = None
    LIST_EXTENSION_BASE = []

    for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
        if EACH_LINE.strip().startswith('EXTENSIONS:'):
            INDEX_BASE_EXTENSION = EACH_INDEX_LINE
            STRING_AFTER = EACH_LINE.split(':', 1)[1].strip()
            if STRING_AFTER and STRING_AFTER != '-':
                LIST_EXTENSION_BASE = [EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_AFTER.split(',') if EACH_EXTENSION.strip()]
            break

    LIST_EXTENSION_ADD = []

    if EACH_REGISTER in MAP_SECTION_EXTENSION:
        MATCH_EXTENSION = re.search(r'^\s*EXTENSIONS:\s*(.+)$', MAP_SECTION_EXTENSION[EACH_REGISTER], flags=re.M)
        if MATCH_EXTENSION:
            LIST_EXTENSION_ADD = [EACH_EXTENSION.strip() for EACH_EXTENSION in MATCH_EXTENSION.group(1).split(',') if EACH_EXTENSION.strip()]

    LIST_EXTENSION_MERGED = sorted(set(LIST_EXTENSION_BASE + LIST_EXTENSION_ADD))
    STRING_EXTENSION_LINE = 'EXTENSIONS: ' + (', '.join(LIST_EXTENSION_MERGED) if LIST_EXTENSION_MERGED else '-')

    if INDEX_BASE_EXTENSION is None:
        LIST_LINE.append(STRING_EXTENSION_LINE)
    else:
        LIST_LINE[INDEX_BASE_EXTENSION] = STRING_EXTENSION_LINE

    if EACH_REGISTER in MAP_SECTION_RULE:
        LIST_RULE_LINE = []

        for EACH_RAW in MAP_SECTION_RULE[EACH_REGISTER].splitlines():
            STRING_RAW = EACH_RAW.strip()

            if STRING_RAW.upper().startswith('RULE '):
                STRING_RULE_ORIGINAL = STRING_RAW.replace('“', '"').replace('”', '"')
                EACH_RULE = STRING_RULE_ORIGINAL.split(':', 1)[1].strip()

                LIST_NOT_IMPLEMENTED = ['sireg', 'sireg2', 'sireg3', 'sireg4', 'sireg5', 'sireg6', 'vsireg', 'vsireg2', 'vsireg3', 'vsireg4', 'vsireg5', 'vsireg6', 'jvt', 'mnstatus', 'mncause', 'tcontrol', 'mireg', 'mireg2', 'mireg3', 'mireg4', 'mireg5', 'mireg6', 'mtopei', *[f'hpmcounter{i}' for i in range(19, 32)], *[f'hpmcounter{i}h' for i in range(3, 32)], *[f'mhpmcounter{i}' for i in range(19, 32)], *[f'mhpmcounter{i}h' for i in range(3, 32)], 'pmpcfg1', 'pmpcfg3', 'pmpcfg5', 'pmpcfg7', 'pmpcfg9', 'pmpcfg11', 'pmpcfg13', 'pmpcfg15', 'mctrctl', 'mcyclecfg', 'mcyclecfgh', 'mcycleh', 'medelegh', 'menvcfgh', *[f'mhpmevent{i}h' for i in range(3, 32)], 'midelegh', 'mieh', 'minstretcfg', 'minstretcfgh', 'minstreth', 'mintstatus', 'miph', 'mnepc', 'mnscratch', 'mnxti', 'mscratchcsw', 'mscratchcswl', 'mseccfgh', 'mstatush', 'mtvt', 'mvienh', 'mviph', 'hcontext', 'hedelegh', 'henvcfgh', 'hidelegh', 'htimedeltah', 'hvienh', 'hviph', 'hviprio1h', 'hviprio2h', 'mscontext', 'scontext', 'sctrctl', 'sctrdepth', 'sctrstatus', 'sieh', 'sintstatus', 'siph', 'snxti', 'srmcfg', 'sscratchcsw', 'sscratchcswl', 'stimecmph', 'stopei', 'stvt', 'cycleh', 'instreth', 'timeh', 'uintstatus', 'unxti', 'uscratchcsw', 'uscratchcswl', 'utvt', 'vsieh', 'vsiph', 'vstimecmph', 'vstopei', 'scountovf', 'stimecmp', 'vstimecmp', 'hstateen0h', 'hstateen1h', 'hstateen2h', 'hstateen3h', 'mstateen0h', 'mstateen1h', 'mstateen2h', 'mstateen3h', 'aes32dsi', 'aes32dsmi', 'aes32esi', 'aes32esmi']
                LIST_NOT_IMPLEMENTED += ['tdata3', 'tselect', 'hgeie', 'hgeip']

                STRING_NOT_IMPLEMENTED = ', '.join(LIST_NOT_IMPLEMENTED)

                STRING_RULE_LOWER = EACH_RULE.lower()

                LIST_FORCE_NO_KEYWORD = [
                    'sireg', 'vsireg',
                    'scountovf',
                    'stimecmp', 'vstimecmp',
                    'vs-mode', 'vu-mode',
                    'virtual-instruction',
                    'guest-page fault', 'guest page fault',
                    'hgatp', 'vsatp',
                    'mtval2', 'htval', 'mtinst', 'htinst',
                ]
                
                IS_FORCE_NO = any(EACH_KEYWORD in STRING_RULE_LOWER for EACH_KEYWORD in LIST_FORCE_NO_KEYWORD)
                
                if IS_FORCE_NO:
                    LIST_RULE_CURRENT = []
                    continue

                if '[' in STRING_RULE_ORIGINAL and ']' in STRING_RULE_ORIGINAL:
                    LIST_RULE_LINE.append(STRING_RULE_ORIGINAL)
                    continue

                STRING_PROMPT = f'''You are given one RISC-V architectural register rule. Output YES or NO only. No additional words.

Kernel info:
XLEN = 64
MXLEN = 64
SXLEN = 64
UXLEN = 64
HSXLEN = 64
RV64
M-mode, S-mode, U-mode, and HS-mode are implemented and supported.
U-mode is implemented and supported.
S-mode is implemented and supported.
HS-mode is implemented and supported.
VS-mode and VU-mode are implemented and supported.
Most standard RISC-V extensions and registers are implemented.

Implemented extensions include:
rv64imafdcvh, zicsr, zicntr, zihpm, smstateen, smaia, sscofpmf, sstc, zicfiss, zicbom, zicboz, zca, zimop, zcmop, zbb, zbs, zkne, zbkb, zabha, zacas, zawrs, zkr, smepmp, zcb, zicond, zba, zknd, zbc, zbkc, zfh, zfbfmin, zfhmin, zfa, zifencei, zvfbfmin, zbkx, zvksed, zvksh, zvknha, zvknhb, zvkg, zvfbfwma, zvbc, zvbb, zvkned, zksed, zksh, zknh, zvkb, zicbop, zicfilp, svinval, zve32f.

These registers are not implemented:
{STRING_NOT_IMPLEMENTED}

The test harness can observe integer registers, CSRs, trap cause such as mcause/scause, privilege-mode trap behavior, and custom read-only/write-only CSR side effects.

Output YES if the register rule can be tested by a normal architectural assembly test with an observable result such as:
- reading the CSR value
- writing the CSR and reading the resulting value
- checking whether a CSR access traps
- checking mcause/scause after an illegal CSR access
- checking WARL/WLRL readback behavior
- checking whether read-only or write-only CSR side effects happen
- checking privilege-mode access behavior
- checking whether a lower privilege mode is permitted or denied access to a CSR
- checking counter access permission controlled by mcounteren, scounteren, or hcounteren

For register rules, prefer YES when the behavior is architecturally visible through CSR reads, CSR writes, traps, permission checks, or readback values.

Important applicability rules:
- If the rule references only implemented counters such as cycle, time, instret, hpmcounter3 through hpmcounter18, or their controlling CSRs, do not reject it merely because some other counters are not implemented.
- If the rule references a specific register in the not-implemented list, output NO unless the rule is explicitly about illegal access to an unimplemented register.
- If the rule uses "next implemented privilege mode", treat it as applicable if that next mode exists in this kernel. Since S-mode is implemented, rules about permission in the next implemented privilege mode are testable in S-mode.

Output NO only if the rule is mainly about:
- WPRI/reserved bits whose exact value is not architecturally meaningful
- optional/permissive behavior such as may, might, can, or implementation-defined
- debug/trace/performance behavior not directly visible to normal architectural tests
- physical TLB/cache/page-table synchronization effects
- behavior that cannot be directly observed through architectural state

For register rules:
- Output NO if the rule requires reading or writing any register listed in the not-implemented register list.
- Output NO for rules requiring sireg*, vsireg*, siselect, or vsiselect if those registers are listed as not implemented.
- Example: a rule saying a counter can be read or written via sireg* should be NO when sireg* is not implemented.
- Output NO if the rule only says a register exists and its fields are WARL or may be read-only zero, without a concrete required readback value or trap behavior.
- Example: "In harts with U-mode, mcounteren must be implemented, but all fields are WARL and may be read-only zero" should be NO.
- Output YES for WARL rules only when the rule states a concrete forbidden value, required masking behavior, or required readback constraint.
- For counter-inhibit registers, output YES for rules saying that a counter increments when its inhibit bit is clear.
- For counter-inhibit registers, output YES for rules saying that a counter does not increment when its inhibit bit is set.
- Apply this to CY, IR, and HPM_n_ fields consistently.
- For counter-enable registers such as mcounteren, output YES for both clear-bit deny rules and set-bit permit rules for CY, TM, IR, and HPMn.
- Example: "When CY is clear, reading cycle in S-mode or U-mode causes illegal-instruction exception" should be YES.
- Output NO if the rule's main behavior is about a different CSR than the current register.
- Example: if the current register is mcountinhibit, a rule mainly about scountinhibit readback should be NO.
- Output NO for rules whose main observable effect requires memory ordering, page-table behavior, address translation, device I/O ordering, or cache/block-operation semantics.
- Output YES for menvcfg only when the rule can be tested through CSR readback, CSR access trap/no-trap behavior, or direct illegal-instruction behavior of a simple instruction.

For register rules involving trap delegation:
- Output YES for simple M/S/U trap delegation behavior that can be tested without virtualization.
- Output NO for guest-page-fault delegation rules or rules requiring hypervisor guest translation behavior.
- Output NO if the rule requires guest-page faults.

The register:
{EACH_REGISTER}

The rule:
{EACH_RULE}'''

                if IS_LOCAL:
                    RESPONSE = chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content
                else:
                    RESPONSE = CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content

                if RESPONSE.strip().upper().startswith('YES'):
                    LIST_RULE_LINE.append(STRING_RULE_ORIGINAL)

        if LIST_RULE_LINE:
            NUMBER_RULE = 1
            LIST_LINE_WITHOUT_RULE = []

            for EACH_LINE in LIST_LINE:
                if not re.match(r'^\s*RULE\s+\d+\s*:', EACH_LINE, flags=re.I):
                    LIST_LINE_WITHOUT_RULE.append(EACH_LINE)

            LIST_LINE = LIST_LINE_WITHOUT_RULE

            for EACH_RULE in LIST_RULE_LINE:
                STRING_RULE = EACH_RULE.split(':', 1)[1].strip()
                LIST_LINE.append(f'RULE {NUMBER_RULE}: {STRING_RULE}')
                NUMBER_RULE += 1

    if EACH_REGISTER not in MAP_SECTION_RULE:
        continue

    LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE).strip())

LIST_BLOCK_FINAL = FUNCTION_REWRITE_FINAL_REGISTER_RULES(LIST_BLOCK_FINAL)

def FUNCTION_REMOVE_EMPTY_REGISTER_BLOCKS(LIST_BLOCK):
    LIST_OUTPUT_BLOCK = []

    for EACH_BLOCK in LIST_BLOCK:
        HAS_REGISTER = any(
            EACH_LINE.strip().startswith('REGISTER:')
            for EACH_LINE in EACH_BLOCK.splitlines()
        )

        HAS_RULE = any(
            re.match(r'^\s*RULE\s+\d+\s*:', EACH_LINE)
            for EACH_LINE in EACH_BLOCK.splitlines()
        )

        if (not HAS_REGISTER) or HAS_RULE:
            LIST_OUTPUT_BLOCK.append(EACH_BLOCK.strip())

    return LIST_OUTPUT_BLOCK

LIST_BLOCK_FINAL = FUNCTION_REMOVE_EMPTY_REGISTER_BLOCKS(LIST_BLOCK_FINAL)

Path(FILE_OUTPUT).write_text('\n\n'.join(LIST_BLOCK_FINAL).strip() + '\n', encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 11 - UPDATE 2 - 7_REGISTER_FINAL.txt - END

9 hour(s) 29 minute(s) 38.33 second(s)

====
DONE
====


In [12]:
# 12 - CREATE - 8_REGISTER_TOTAL.txt - START

"""
Maintain a cumulative collection of RISC-V CSRs, metadata, and architectural rules by merging new results, replacing outdated entries, and sorting registers alphabetically.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/8_REGISTER_TOTAL.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_INPUT = FILE_HANDLE.read()

if (not os.path.exists(FILE_OUTPUT)) or os.path.getsize(FILE_OUTPUT) == 0:
    TEXT_OUTPUT = TEXT_INPUT
else:
    with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_PREVIOUS = FILE_HANDLE.read()
    TEXT_COMBINED = TEXT_PREVIOUS.strip() + '\n\n' + TEXT_INPUT.strip()
    REGEX_BLOCK = re.compile(r'(?ms)^\s*REGISTER:\s*([^\n]+)\s*\n.*?(?=^\s*REGISTER:\s*[^\n]+\s*\n|\Z)')
    MAP_BLOCK = {MATCH.group(1).strip(): MATCH.group(0) for MATCH in REGEX_BLOCK.finditer(TEXT_COMBINED)}
    TEXT_OUTPUT = '\n\n'.join(MAP_BLOCK[EACH_NAME].strip() for EACH_NAME in sorted(MAP_BLOCK, key=str.upper)).strip()

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(TEXT_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 12 - CREATE - 8_REGISTER_TOTAL.txt - END

0.02 second(s)

====
DONE
====


In [ ]:
# 13 - CREATE - 9_REGISTER_SUMMARY.txt - START

"""
Generate concise summaries of RISC-V CSR semantics from extracted architectural rules using LLMs, with chunk-based summarization and merging for large rule sets.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from typing import Dict, List, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/8_REGISTER_TOTAL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/9_REGISTER_SUMMARY.txt'
FILE_PROMPT_1 = '5_REGISTER/2_HISTORY/7_HISTORY_REGISTER_SUMMARY_1.txt'
FILE_PROMPT_2 = '5_REGISTER/2_HISTORY/8_HISTORY_REGISTER_SUMMARY_2.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_SANITIZE_TEXT(STRING_VALUE: str) -> str:
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'").strip())

def FUNCTION_CALL_LLM(STRING_PROMPT: str) -> Tuple[str, str]:
    if IS_LOCAL:
        STRING_RAW = chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content
    else:
        STRING_RAW = CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content
    return FUNCTION_SANITIZE_TEXT(STRING_RAW), STRING_RAW

def FUNCTION_PARSE_REGISTER_RULE(TEXT_VALUE: str) -> Dict[str, List[Tuple[int, str]]]:
    REGISTER_CURRENT = None
    MAP_REGISTER: Dict[str, List[Tuple[int, str]]] = {}
    for EACH_LINE in TEXT_VALUE.splitlines():
        MATCH_REGISTER = REGEX_REGISTER.match(EACH_LINE)
        if MATCH_REGISTER:
            REGISTER_CURRENT = MATCH_REGISTER.group(1).strip()
            MAP_REGISTER.setdefault(REGISTER_CURRENT, [])
            continue
        MATCH_RULE = REGEX_RULE.match(EACH_LINE)
        if MATCH_RULE and REGISTER_CURRENT is not None:
            MAP_REGISTER[REGISTER_CURRENT].append((int(MATCH_RULE.group(1)), MATCH_RULE.group(2).strip()))
    for EACH_REGISTER_NAME in MAP_REGISTER:
        MAP_REGISTER[EACH_REGISTER_NAME].sort(key=lambda EACH_X: EACH_X[0])
    return MAP_REGISTER

def FUNCTION_LOG_BLOCK(STRING_HISTORY: str, GLOBAL_NUMBER: int, LOCAL_NUMBER: int, STRING_NAME: str, STRING_PROMPT: str, STRING_RESPONSE: str) -> str:
    STRING_HALF = '=' * (WIDTH_PROMPT // 2)
    STRING_INTRO = (f' START - {GLOBAL_NUMBER} '.center(WIDTH_PROMPT, '=') + '\n' + f'GLOBAL NUMBER: {GLOBAL_NUMBER}\n' + f'LOCAL NUMBER: {LOCAL_NUMBER}\n' + f'NAME: {STRING_NAME}\n' + f'{STRING_HALF}\n' + '=======\n' + 'PROMPT:\n' + '=======\n' + f'{STRING_HALF}\n')
    STRING_ENDING = (f'\n{STRING_HALF}\n' + '=========\n' + 'RESPONSE:\n' + '=========\n' + f'{STRING_HALF}\n' + f'{STRING_RESPONSE}\n')
    STRING_OUTRO = f' END - {GLOBAL_NUMBER} '.center(WIDTH_PROMPT, '=') + STRING_SUFFIX_PROMPT
    return STRING_HISTORY + STRING_INTRO + STRING_PROMPT + STRING_ENDING + STRING_OUTRO

def FUNCTION_SUMMARIZE_REGISTER(REGISTER_NAME: str, LIST_RULE: List[Tuple[int, str]]):
    global STRING_PROMPT_HISTORY_1, GLOBAL_COUNTER_1, LOCAL_COUNTER_1
    LIST_CHUNK = [LIST_RULE[EACH_INDEX:EACH_INDEX + CHUNK_SIZE] for EACH_INDEX in range(0, len(LIST_RULE), CHUNK_SIZE)]
    LIST_PARTIAL: List[str] = []
    for EACH_INDEX_CHUNK, EACH_LIST_CHUNK in enumerate(LIST_CHUNK, 1):
        STRING_RULE_TEXT = '\n'.join([f'RULE {EACH_RULE_NUMBER}: {EACH_RULE_TEXT}' for EACH_RULE_NUMBER, EACH_RULE_TEXT in EACH_LIST_CHUNK])
        STRING_PROMPT = STRING_PROMPT_CHUNK.format(register=REGISTER_NAME, rules_text=STRING_RULE_TEXT)
        STRING_CLEANED, STRING_RAW = FUNCTION_CALL_LLM(STRING_PROMPT)
        STRING_PROMPT_HISTORY_1 = FUNCTION_LOG_BLOCK(STRING_PROMPT_HISTORY_1, GLOBAL_COUNTER_1, LOCAL_COUNTER_1, REGISTER_NAME, STRING_PROMPT, STRING_RAW)
        GLOBAL_COUNTER_1 += 1
        LOCAL_COUNTER_1 += 1
        LIST_PARTIAL.append(STRING_CLEANED)
    if len(LIST_PARTIAL) == 1:
        STRING_MERGE_PROMPT = STRING_PROMPT_MERGE.format(register=REGISTER_NAME, partials=f'- {LIST_PARTIAL[0]}')
        STRING_FINAL, STRING_RAW_2 = FUNCTION_CALL_LLM(STRING_MERGE_PROMPT)
        return LIST_PARTIAL[0], STRING_MERGE_PROMPT, STRING_RAW_2
    STRING_MERGE_PROMPT = STRING_PROMPT_MERGE.format(register=REGISTER_NAME, partials='\n\n'.join(f'- {EACH_PARTIAL}' for EACH_PARTIAL in LIST_PARTIAL))
    STRING_FINAL, STRING_RAW_2 = FUNCTION_CALL_LLM(STRING_MERGE_PROMPT)
    return STRING_FINAL, STRING_MERGE_PROMPT, STRING_RAW_2
# FUNCTION - FINISH

# VARIABLE - BEGIN
CHUNK_SIZE = 10000
REGEX_REGISTER = re.compile(r'^\s*REGISTER:\s*(.+?)\s*$')
REGEX_RULE = re.compile(r'^\s*RULE\s+(\d+)\s*:\s*(.+?)\s*$')
WIDTH_PROMPT = 120
STRING_SUFFIX_PROMPT = '\n' + (('#' * WIDTH_PROMPT + '\n') * 3).rstrip('\n') + '\n'
STRING_PROMPT_HISTORY_1 = ''
STRING_PROMPT_HISTORY_2 = ''
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
LIST_OUTPUT_LINE: List[str] = []
STRING_PROMPT_CHUNK = '''You are summarizing RISC-V register semantics strictly from the provided rules.

Register: {register}

Rules (subset):
{rules_text}

Task:
Write a concise technical summary (maximum 4 sentences) capturing only the high-level semantics explicitly stated in the rules.

Focus on:
- register purpose
- key fields (group similar fields together)
- access control and privilege restrictions
- notable exception behavior

Rules:
- Do NOT list every field bit-by-bit.
- Do NOT enumerate rule numbers.
- Do NOT include redundant or repetitive details.
- Group related behaviors into one sentence.
- Omit low-level details (e.g., exact bit positions, aliases, mirrored bits) unless they are essential.
- Prefer clarity over completeness.

Output must be ASCII-only text.
Return only the paragraph.'''
STRING_PROMPT_MERGE = '''You are merging partial summaries of the same RISC-V register derived from rule subsets.

Register: {register}

Partial summaries:
{partials}

Task:
Produce a single cohesive summary (4-6 sentences, one paragraph) that preserves all semantics present in the partial summaries, including fields, access control, privilege/virtualization constraints, and exception behavior.
- Do NOT introduce new semantics not present in the partial summaries.
- Do NOT remove constraints, field conditions, or exception cases.
- Remove redundancy by merging equivalent statements only.
- Prefer canonical RISC-V wording and precise terminology.
Remember, output must be ASCII-only text.
Return only the final paragraph.'''
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    MAP_REGISTER = FUNCTION_PARSE_REGISTER_RULE(FILE_HANDLE.read())

for EACH_REGISTER_NAME, EACH_LIST_RULE in MAP_REGISTER.items():
    STRING_SUMMARY, STRING_MERGE_PROMPT, STRING_MERGE_RAW = FUNCTION_SUMMARIZE_REGISTER(EACH_REGISTER_NAME, EACH_LIST_RULE)
    LIST_OUTPUT_LINE += [f'REGISTER: {EACH_REGISTER_NAME}', f'SUMMARY: {STRING_SUMMARY}', '']
    global STRING_PROMPT_HISTORY_2, GLOBAL_COUNTER_2, LOCAL_COUNTER_2
    STRING_PROMPT_HISTORY_2 = FUNCTION_LOG_BLOCK(STRING_PROMPT_HISTORY_2, GLOBAL_COUNTER_2, LOCAL_COUNTER_2, EACH_REGISTER_NAME, STRING_MERGE_PROMPT, STRING_MERGE_RAW)
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_1.removesuffix(STRING_SUFFIX_PROMPT))

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_2.removesuffix(STRING_SUFFIX_PROMPT))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n'.join(LIST_OUTPUT_LINE).strip())
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 13 - CREATE - 9_REGISTER_SUMMARY.txt - END

31 minute(s) 56.14 second(s)

====
DONE
====
